<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/02_stageII_esen_oc25_adsorbml_relaxation_CHE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/isocan/ML-accelerated-ORR/blob/main/notebooks/02_stageII_esen_oc25_adsorbml_relaxation_CHE.ipynb"
   target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Open in Colab"/>
</a>

# Stage II — eSEN/OC25 AdsorbML Relaxation and CHE Adsorption Energies

This Google Colab notebook is a reproducibility tutorial for the eSEN branch of
Stage II in the multi-fidelity oxygen reduction reaction workflow. The complete
study applied the same procedure to 24 top candidates selected after Stage I.
For clarity and manageable runtime, this public notebook demonstrates the full
workflow for one selected slab:

```python
SURFACE_ID = "mp-10260_111"
```

## Workflow

1. Install and verify the FAIRChem environment.
2. Load the `esen-sm-conserving-all-oc25` checkpoint.
3. Generate the selected slab and O*, OH*, and OOH* adsorbates.
4. Relax the bare slab first and record its eSEN total energy.
5. Relax H₂ and H₂O with the same model to define the CHE references.
6. Generate AdsorbML heuristic and random adsorption configurations.
7. Relax every adsorbate–slab structure with eSEN/OC25.
8. Calculate the electronic adsorption energy as

$$
E_{\mathrm{ads}}^{\mathrm{eSEN/CHE}}
=E_{\mathrm{ads+slab}}-E_{\mathrm{slab}}-E_{\mathrm{reference}}.
$$

9. Validate trajectories, classify adsorption sites, and select unique-site and
   global minima.
10. Export the bare slab and global O*, OH*, and OOH* minima as RPBE-D3
    single-point VASP calculations.

## CHE reference energies

$$
E_{\mathrm{reference}}(\mathrm O^*)
=E_{\mathrm{H_2O}}-E_{\mathrm{H_2}},
$$

$$
E_{\mathrm{reference}}(\mathrm{OH}^*)
=E_{\mathrm{H_2O}}-\frac{1}{2}E_{\mathrm{H_2}},
$$

$$
E_{\mathrm{reference}}(\mathrm{OOH}^*)
=2E_{\mathrm{H_2O}}-\frac{3}{2}E_{\mathrm{H_2}}.
$$

The derived adsorption energy is stored once, in the column
`eSEN_pred_ads_energy_eV`. The raw eSEN total energies and the CHE reference
term are retained separately for traceability.

## Scientific scope

This notebook reports electronic adsorption energies only. It does not apply
ZPE, entropy, solvation, limiting-potential, or overpotential corrections.
Those quantities are evaluated after the subsequent RPBE-D3, VaspGibbs, and
VASPsol calculations.

The molecular references are evaluated with the same eSEN/OC25 checkpoint to
maintain a common model energy zero. Because isolated molecules are outside the
main OC25 interface domain, the eSEN adsorption energies are screening values
that require DFT validation.

## VASP export

The exported folders are prepared for non-spin-polarized RPBE-D3 single-point
calculations. The in-plane Monkhorst–Pack mesh follows

$$
N_a=\max\left(1,\operatorname{round}\frac{40}{|\mathbf a|}\right),\qquad
N_b=\max\left(1,\operatorname{round}\frac{40}{|\mathbf b|}\right),\qquad
N_c=1.
$$

Licensed VASP PAW datasets are not redistributed. Each calculation directory
contains a simple `POTCAR` placeholder that states the required element order.


## 1. Reproducible Colab environment

Select **Runtime → Change runtime type → T4 GPU** before running the notebook.

Start from a fresh Colab runtime:

1. run the installation cell;
2. run the restart cell once;
3. after Colab reconnects, run the notebook again from the top.

Simple marker files prevent repeated installation and restart loops.


In [1]:
# Install once per fresh Colab runtime.
#
# Important:
# - Do not import NumPy, SciPy, pandas, ASE, or FAIRChem before the restart.
# - The packages below are installed together so pip can solve one
#   consistent environment.

from pathlib import Path
import subprocess
import sys

ENV_MARKER = Path("/content/.stageII_esen_oc25_environment_installed")
RESTART_MARKER = Path("/content/.stageII_esen_oc25_kernel_restarted")

if ENV_MARKER.exists():
    print("Compatible Stage II environment is already installed.")
else:
    bootstrap_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "pip",
        "wheel",
        "setuptools<81",
    ]

    environment_command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--upgrade",
        "--prefer-binary",

        # Binary scientific stack compatible with fairchem-core 2.21.0.
        "numpy==2.2.6",
        "scipy==1.16.3",
        "pandas==2.3.2",
        "ase==3.26.0",

        # FAIRChem v2 and Open Catalyst structure-generation utilities.
        "fairchem-core==2.21.0",
        "fairchem-data-oc==1.0.2",

        # Notebook utilities.
        "huggingface-hub>=0.30,<1",
        "py3Dmol>=2.4,<3",
        "matplotlib>=3.9,<4",
        "ipywidgets>=8,<9",
        "tqdm>=4.66",
    ]

    for command in (bootstrap_command, environment_command):
        print("\nRunning:")
        print(" ".join(command))

        try:
            subprocess.check_call(command)
        except subprocess.CalledProcessError:
            print("\nInstallation failed while running:")
            print(" ".join(command))
            print(
                "\nThe full pip resolver output is shown above. "
                "Do not continue to the import cells."
            )
            raise

    print("\nRunning pip check:")
    check_result = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        text=True,
        capture_output=True,
    )

    if check_result.stdout.strip():
        print(check_result.stdout.strip())

    if check_result.stderr.strip():
        print(check_result.stderr.strip())

    if check_result.returncode != 0:
        print(
            "\nNote: pip check reported one or more conflicts. "
            "If they concern unrelated Colab packages that are not imported "
            "by this notebook, the Stage II workflow may still be usable."
        )

    ENV_MARKER.write_text("installed\n", encoding="utf-8")
    RESTART_MARKER.unlink(missing_ok=True)

    print("\nInstallation completed successfully.")
    print("Now run the next cell once to restart the Python kernel.")



Running:
/usr/bin/python3 -m pip install --upgrade pip wheel setuptools<81

Running:
/usr/bin/python3 -m pip install --no-cache-dir --upgrade --prefer-binary numpy==2.2.6 scipy==1.16.3 pandas==2.3.2 ase==3.26.0 fairchem-core==2.21.0 fairchem-data-oc==1.0.2 huggingface-hub>=0.30,<1 py3Dmol>=2.4,<3 matplotlib>=3.9,<4 ipywidgets>=8,<9 tqdm>=4.66

Running pip check:
google-colab 1.0.0 has requirement pandas==2.2.2, but you have pandas 2.3.2.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
gradio 6.20.0 has requirement huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2.
pytensor 2.38.3 has requirement numba<=0.65.1,>=0.58, but you have numba 0.66.0.
cuml-cu12 26.2.0 has requirement numba<0.62.0,>=0.60.0, but you have numba 0.66.0.
torchvision 0.26.0+cu128 has requirement torch==2.11.0, but you have torch 2.8.0.
cudf-cu12 26.2.1 has requirement numba<0.62.0,>=0.60.0, but you have numba 0.66.0.
transformers 5.13.1 has requirement huggingface-h

In [ ]:
# Required one-time kernel restart after installing the binary stack.

from pathlib import Path
import os
import time

ENV_MARKER = Path("/content/.stageII_esen_oc25_environment_installed")
RESTART_MARKER = Path("/content/.stageII_esen_oc25_kernel_restarted")

if not ENV_MARKER.exists():
    raise RuntimeError(
        "The environment installation did not complete. "
        "Run the installation cell and inspect its full pip output."
    )

if RESTART_MARKER.exists():
    print("Kernel restart already completed. Continue below.")
else:
    RESTART_MARKER.write_text("restarted\n", encoding="utf-8")
    print("Restarting the Colab kernel now...")
    time.sleep(1)
    os.kill(os.getpid(), 9)


Restarting the Colab kernel now...


In [1]:
# Verify the environment only after the kernel restart.

import importlib.metadata as metadata
import sys

import ase
import numpy as np
import pandas as pd
import scipy
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("pandas:", pd.__version__)
print("ASE:", ase.__version__)
print("fairchem-core:", metadata.version("fairchem-core"))
print("fairchem-data-oc:", metadata.version("fairchem-data-oc"))

assert np.__version__ == "2.2.6"
assert scipy.__version__ == "1.16.3"
assert pd.__version__ == "2.3.2"
assert ase.__version__ == "3.26.0"
assert torch.__version__.split("+")[0].startswith("2.8.")
assert metadata.version("fairchem-core") == "2.21.0"
assert metadata.version("fairchem-data-oc") == "1.0.2"

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Select Runtime → Change runtime type → T4 GPU."
    )

print("\nEnvironment verification passed.")


Python: 3.12.13
PyTorch: 2.8.0+cu128
CUDA build: 12.8
CUDA available: True
NumPy: 2.2.6
SciPy: 1.16.3
pandas: 2.3.2
ASE: 3.26.0
fairchem-core: 2.21.0
fairchem-data-oc: 1.0.2

Environment verification passed.


## 2. Hugging Face authentication

The OC25 checkpoint is distributed through the gated `facebook/OC25`
repository. Request model access, create a read token, and store the token
in Colab under **Secrets** with the name `HF_TOKEN`.

The token is used only for checkpoint download and is not written to the
result directory or ZIP archive.


In [2]:
import os
from huggingface_hub import login, notebook_login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = hf_token
    print("Authenticated using the HF_TOKEN Colab secret.")
else:
    print("No HF_TOKEN secret found. Complete the interactive login.")
    notebook_login()


Authenticated using the HF_TOKEN Colab secret.


## 3. Imports and calculation settings

Select one Stage I candidate by editing `SURFACE_ID`. The text before the
final underscore is the Materials Project bulk identifier, and the
three-digit suffix is interpreted as the Miller index.

Examples:

```python
SURFACE_ID = "mp-10260_111"
SURFACE_ID = "mp-1078755_110"
SURFACE_ID = "mp-12086_100"
```

`SURFACE_INDEX` selects a termination when the slab generator returns more
than one symmetry-distinct termination for the requested facet.


In [3]:
from __future__ import annotations

import io
import itertools
import json
import random
import re
import shutil
import textwrap
import time
import warnings
from pathlib import Path

import ase.io as aseio
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
import torch

from ase import Atoms
from ase.calculators.singlepoint import SinglePointCalculator
from ase.constraints import FixAtoms
from ase.data import atomic_numbers, covalent_radii
from ase.optimize import BFGS
from IPython.display import clear_output, display
from tqdm.auto import tqdm

from fairchem.core import FAIRChemCalculator, pretrained_mlip
from fairchem.data.oc.core import Adsorbate, AdsorbateSlabConfig, Bulk, Slab
from fairchem.data.oc.utils import DetectTrajAnomaly

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning, module='ase')

# Candidate selection
SURFACE_ID = 'mp-10260_111'
SURFACE_INDEX = 0

# eSEN/OC25 relaxation
MODEL_NAME = 'esen-sm-conserving-all-oc25'
ADSORBATE_NAMES = ('O', 'OH', 'OOH')
ADSORBATE_SIZES = {'O': 1, 'OH': 2, 'OOH': 3}
RANDOM_SITES_PER_ADSORBATE = 20
FMAX_THRESHOLD = 0.05
MAX_RELAX_STEPS = 400
OVERWRITE_EXISTING = False


# ---------------------------------------------------------------------
# CHE molecular references for electronic adsorption energies
# ---------------------------------------------------------------------

REFERENCE_CELL_A = 20.0
REFERENCE_FMAX_THRESHOLD = 0.03
REFERENCE_MAX_RELAX_STEPS = 200
REFERENCE_OVERWRITE_EXISTING = False

# Stage II stops at electronic CHE adsorption energies. ZPE, entropy,
# solvation, limiting potential, and overpotential are evaluated later
# from the DFT + VaspGibbs/VASPsol results.

# VASP export: non-spin-polarized RPBE-D3 single-point calculation
# Stage I convention: round(40/|a|), round(40/|b|), 1.
KPOINT_DENSITY = 40.0
VASP_RUN_MODE = "Single Point"
VASP_PREC = 'Normal'
VASP_ENCUT_EV = 400
VASP_EDIFF = 1e-6
VASP_EDIFFG = -0.02
VASP_NELM = 300
VASP_NSW = 1
VASP_NCORE = 16
VASP_KPAR = 4
VASP_IVDW = 11

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ROOT_DIR = Path('/content/stageII_esen_oc25_results')
ROOT_DIR.mkdir(parents=True, exist_ok=True)

print('Selected surface:', SURFACE_ID)
print('Model:', MODEL_NAME)
print('Force threshold:', FMAX_THRESHOLD, 'eV/Å')
print('VASP k-point rule: round(40/|a|), round(40/|b|), 1')


Selected surface: mp-10260_111
Model: esen-sm-conserving-all-oc25
Force threshold: 0.05 eV/Å
VASP k-point rule: round(40/|a|), round(40/|b|), 1


In [4]:
selected_surface = SURFACE_ID.strip()


def parse_surface_id(
    surface_id: str,
) -> tuple[str, tuple[int, int, int]]:
    """Parse a surface identifier such as mp-10260_111."""
    try:
        bulk_id, miller_text = surface_id.rsplit('_', 1)
    except ValueError as exc:
        raise ValueError(
            "Use the form 'mp-id_facet', for example 'mp-10260_111'."
        ) from exc

    if not bulk_id.startswith('mp-'):
        raise ValueError(f'Invalid Materials Project ID: {bulk_id}')

    if len(miller_text) != 3 or not miller_text.isdigit():
        raise ValueError(
            'Use a three-digit positive facet suffix such as 111, 110, or 100.'
        )

    return bulk_id, tuple(int(value) for value in miller_text)


BULK_ID, MILLER_INDEX = parse_surface_id(selected_surface)
SYSTEM_ID = (
    f"{BULK_ID}_{''.join(map(str, MILLER_INDEX))}"
    f'_term{SURFACE_INDEX}_esen_oc25'
)

SYSTEM_DIR = ROOT_DIR / SYSTEM_ID
TRAJECTORY_DIR = SYSTEM_DIR / 'trajectories'
LOG_DIR = SYSTEM_DIR / 'logs'
POSCAR_DIR = SYSTEM_DIR / 'poscars'
TABLE_DIR = SYSTEM_DIR / 'tables'
VASP_DIR = SYSTEM_DIR / 'vasp'

for directory in (
    SYSTEM_DIR,
    TRAJECTORY_DIR,
    LOG_DIR,
    POSCAR_DIR,
    TABLE_DIR,
    VASP_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print('Materials Project bulk ID:', BULK_ID)
print('Requested Miller index:', MILLER_INDEX)
print('Selected termination:', SURFACE_INDEX)
print('Output directory:', SYSTEM_DIR)


Materials Project bulk ID: mp-10260
Requested Miller index: (1, 1, 1)
Selected termination: 0
Output directory: /content/stageII_esen_oc25_results/mp-10260_111_term0_esen_oc25


## 4. Load the eSEN/OC25 potential

The Stage II model is the energy-conserving OC25 checkpoint referenced as
`esen-sm-conserving-all-oc25`. The checkpoint is loaded through the
FAIRChem pretrained-model registry and wrapped as an ASE calculator.

Model access and calculator construction are tested before the structure
generation and relaxation steps.


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

predictor = pretrained_mlip.get_predict_unit(
    MODEL_NAME,
    device=device,
)


def make_calculator() -> FAIRChemCalculator:
    """Return the OC25 ASE calculator used for every energy evaluation."""
    return FAIRChemCalculator(
        predictor,
        task_name="oc25",
    )


# Construct one calculator immediately so model-access or API errors appear
# before the expensive relaxation loop.
_ = make_calculator()

print("Loaded:", MODEL_NAME)
print("Device:", device)
print("Calculator task: oc25")


checkpoints/esen_sm_conserve.pt:   0%|          | 0.00/51.1M [00:00<?, ?B/s]

iso_atom_elem_refs.yaml: 0.00B [00:00, ?B/s]

Loaded: esen-sm-conserving-all-oc25
Device: cuda
Calculator task: oc25


## 5. Generate the selected surface

The bulk structure is selected by Materials Project source identifier. The
requested Miller index is passed to the Open Catalyst slab generator, and the
termination specified by `SURFACE_INDEX` is retained.

The Open Catalyst atom tags are preserved throughout the workflow:

- tag 0: constrained subsurface atom;
- tag 1: mobile surface atom;
- tag 2: adsorbate atom.

Atoms with tag 0 are fixed during the eSEN relaxation. The same selective-
dynamics flags are preserved in the exported POSCAR files for provenance,
although the prepared VASP calculations are single-point calculations.


In [6]:
def rebuild_atoms(
    atoms: Atoms,
    *,
    slab: bool = False,
    molecule: bool = False,
) -> Atoms:
    """Recreate ASE atoms while preserving geometry and OC tags.

    Slab and adsorbate–slab systems are explicitly periodic in all three
    directions, matching the working OC25/Kaggle workflow and retaining the
    vacuum already present in the cell.
    """
    tags = np.asarray(atoms.get_tags(), dtype=int)
    rebuilt = Atoms(
        numbers=np.asarray(atoms.get_atomic_numbers(), dtype=int),
        positions=np.asarray(atoms.get_positions(), dtype=float),
        cell=np.asarray(atoms.cell.array, dtype=float),
        pbc=False if molecule else np.asarray(atoms.get_pbc(), dtype=bool),
        tags=tags,
    )
    if slab:
        rebuilt.set_pbc(True)
        fixed = np.flatnonzero(tags == 0)
        if len(fixed):
            rebuilt.set_constraint(FixAtoms(indices=fixed))
    return rebuilt

def ensure_finite_geometry(atoms: Atoms, label: str) -> None:
    if not np.isfinite(atoms.positions).all():
        raise ValueError(f"Non-finite positions in {label}.")
    if not np.isfinite(atoms.cell.array).all():
        raise ValueError(f"Non-finite cell in {label}.")

bulk = Bulk(bulk_src_id_from_db=BULK_ID)
if getattr(bulk, "src_id", BULK_ID) != BULK_ID:
    raise RuntimeError(
        f"Requested {BULK_ID}, but the database returned "
        f"{getattr(bulk, 'src_id', None)}."
    )

bulk.atoms = rebuild_atoms(bulk.atoms)
generated = Slab.from_bulk_get_specific_millers(
    bulk=bulk,
    specific_millers=MILLER_INDEX,
)

if isinstance(generated, Slab):
    slab_candidates = [generated]
elif generated is None:
    slab_candidates = []
else:
    slab_candidates = list(generated)

if not slab_candidates:
    raise RuntimeError(f"No {MILLER_INDEX} slab was generated for {BULK_ID}.")
if not 0 <= SURFACE_INDEX < len(slab_candidates):
    raise IndexError("SURFACE_INDEX is outside the generated termination range.")

raw_slab = slab_candidates[SURFACE_INDEX]
clean_initial = rebuild_atoms(raw_slab.atoms, slab=True)
ensure_finite_geometry(clean_initial, "clean slab")

slab_object = Slab(
    bulk=raw_slab.bulk,
    slab_atoms=clean_initial,
    millers=raw_slab.millers,
    shift=raw_slab.shift,
    top=raw_slab.top,
    oriented_bulk=raw_slab.oriented_bulk,
)

clean_initial_path = SYSTEM_DIR / "clean_slab_initial.traj"
aseio.write(clean_initial_path, clean_initial)

print("Bulk formula:", bulk.atoms.get_chemical_formula())
print("Slab formula:", clean_initial.get_chemical_formula())
print("Generated terminations:", len(slab_candidates))
print("Selected termination:", SURFACE_INDEX)
display(
    pd.Series(clean_initial.get_tags())
    .value_counts()
    .sort_index()
    .rename_axis("OC_tag")
    .reset_index(name="count")
)


Bulk formula: Ni3Sb
Slab formula: Ni36Sb12
Generated terminations: 4
Selected termination: 0


,OC_tag,count
0,0,36
1,1,12


## 6. Construct the ORR intermediates

O* and OH* are requested from the Open Catalyst adsorbate database. Explicit
molecular geometries are used as a defensive fallback if a database lookup
is unavailable. OOH* is constructed explicitly.

In every adsorbate object, the oxygen atom that binds to the surface is
assigned index 0.


In [7]:
def build_h2() -> Atoms:
    """Return an isolated H2 molecule with a 0.74 Å bond length."""
    bond_length = 0.74
    return Atoms(
        "H2",
        positions=[
            [-bond_length / 2.0, 0.0, 0.0],
            [bond_length / 2.0, 0.0, 0.0],
        ],
        pbc=False,
    )


def build_h2o() -> Atoms:
    """Return an isolated H2O molecule near its gas-phase geometry."""
    oh_distance = 0.9572
    angle_rad = np.deg2rad(104.52)

    return Atoms(
        ["O", "H", "H"],
        positions=[
            [0.0, 0.0, 0.0],
            [oh_distance, 0.0, 0.0],
            [
                oh_distance * np.cos(angle_rad),
                oh_distance * np.sin(angle_rad),
                0.0,
            ],
        ],
        pbc=False,
    )


def build_o() -> Atoms:
    return Atoms("O", positions=[[0.0, 0.0, 0.0]], pbc=False)

def build_oh() -> Atoms:
    return Atoms(
        ["O", "H"],
        positions=[[0.0, 0.0, 0.0], [0.97, 0.0, 0.0]],
        pbc=False,
    )

def build_ooh() -> Atoms:
    return Atoms(
        ["O", "O", "H"],
        positions=[
            [0.000, 0.000, 0.000],
            [0.000, 0.000, 1.450],
            [0.940, 0.000, 1.750],
        ],
        pbc=False,
    )

def load_or_build(label: str, fallback: Atoms, binding: list[int]) -> Adsorbate:
    try:
        ads = Adsorbate(adsorbate_smiles_from_db=label)
        ads.atoms = rebuild_atoms(ads.atoms, molecule=True)
        return ads
    except Exception:
        return Adsorbate(
            adsorbate_atoms=fallback,
            adsorbate_binding_indices=binding,
        )

adsorbates = {
    "O": load_or_build("*O", build_o(), [0]),
    "OH": load_or_build("*OH", build_oh(), [0]),
    "OOH": Adsorbate(
        adsorbate_atoms=build_ooh(),
        adsorbate_binding_indices=[0],
    ),
}

adsorbate_summary = pd.DataFrame(
    [
        {
            "adsorbate": name,
            "formula": ads.atoms.get_chemical_formula(),
            "n_atoms": len(ads.atoms),
            "binding_indices": list(ads.binding_indices),
        }
        for name, ads in adsorbates.items()
    ]
)
display(adsorbate_summary)


,adsorbate,formula,n_atoms,binding_indices
0,O,O,1,[0]
1,OH,HO,2,[0]
2,OOH,HO2,3,[0]


## 7. Shared eSEN/OC25 structural-relaxation routine

The same calculator and relaxation routine are used for the bare slab and all
adsorbate–slab systems. Slab atoms with OC tag 0 are fixed, the generated
vacuum cell is retained, and periodic boundary conditions are explicitly
enabled before the eSEN calculation.

Each structure is relaxed with BFGS until

$$
F_{\max}<0.05\ \mathrm{eV\,A^{-1}},
$$

or until `MAX_RELAX_STEPS` is reached. Every calculation is written to an
independent trajectory and log file. When `OVERWRITE_EXISTING=False`, readable
trajectories are reused so interrupted Colab calculations can be resumed.


In [8]:
def maximum_force(forces: np.ndarray) -> float:
    forces = np.asarray(forces, dtype=float)
    return float(np.linalg.norm(forces, axis=1).max()) if forces.size else np.nan

def read_frames(path: Path) -> list[Atoms]:
    frames = aseio.read(str(path), index=":")
    return frames if isinstance(frames, list) else [frames]

def prepare_relaxation_atoms(atoms: Atoms) -> Atoms:
    return rebuild_atoms(atoms, slab=True)

def final_with_results(atoms: Atoms) -> Atoms:
    energy = float(atoms.get_potential_energy())
    forces = np.asarray(atoms.get_forces(), dtype=float)
    final = atoms.copy()
    final.calc = SinglePointCalculator(final, energy=energy, forces=forces)
    return final

def relax_structure(atoms: Atoms, trajectory_path: Path, log_path: Path) -> dict:
    trajectory_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)

    base = {
        "status": "failed",
        "error": "",
        "elapsed_s": np.nan,
        "n_frames": 0,
        "ml_total_energy_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "converged": False,
        "trajectory": str(trajectory_path),
        "log": str(log_path),
    }

    if trajectory_path.exists() and not OVERWRITE_EXISTING:
        try:
            frames = read_frames(trajectory_path)
            final = frames[-1]
            energy = float(final.get_potential_energy())
            fmax = maximum_force(final.get_forces())
            return {
                **base,
                "status": "reused",
                "n_frames": len(frames),
                "ml_total_energy_eV": energy,
                "final_fmax_eV_A": fmax,
                "converged": bool(np.isfinite(fmax) and fmax < FMAX_THRESHOLD),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    working = prepare_relaxation_atoms(atoms)
    working.calc = make_calculator()
    start = time.time()

    try:
        optimizer = BFGS(
            working,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(fmax=FMAX_THRESHOLD, steps=MAX_RELAX_STEPS)

        final = final_with_results(working)
        frames = read_frames(trajectory_path)
        if not frames:
            frames = [final]
        elif np.max(np.abs(frames[-1].positions - final.positions)) > 1e-10:
            frames.append(final)
        else:
            frames[-1] = final
        aseio.write(str(trajectory_path), frames, format="traj")

        energy = float(final.get_potential_energy())
        fmax = maximum_force(final.get_forces())
        return {
            **base,
            "status": "completed",
            "elapsed_s": time.time() - start,
            "n_frames": len(frames),
            "ml_total_energy_eV": energy,
            "final_fmax_eV_A": fmax,
            "converged": bool(np.isfinite(fmax) and fmax < FMAX_THRESHOLD),
        }
    except Exception as exc:
        return {
            **base,
            "elapsed_s": time.time() - start,
            "error": f"{type(exc).__name__}: {exc}",
        }


## 8. Relax the bare slab first

The clean slab is relaxed before any adsorption-energy calculation. Its final
eSEN/OC25 total energy is stored as `E_slab_eV` and reused unchanged for every
O*, OH*, and OOH* configuration on this surface.


In [9]:
clean_trajectory_path = TRAJECTORY_DIR / "bare" / "bare.traj"
clean_log_path = LOG_DIR / "bare" / "bare.log"

# The bare slab is always evaluated before molecular references and
# adsorbate–slab calculations.
clean_relaxation = relax_structure(
    clean_initial,
    clean_trajectory_path,
    clean_log_path,
)

if clean_relaxation["status"] not in {"completed", "reused"}:
    raise RuntimeError(
        "Clean slab relaxation failed: " + clean_relaxation["error"]
    )
if not clean_relaxation["converged"]:
    raise RuntimeError("The clean slab did not reach the force threshold.")

clean_final = aseio.read(clean_relaxation["trajectory"], index=-1)
clean_energy_eV = float(clean_relaxation["ml_total_energy_eV"])

clean_relaxation_table = pd.DataFrame(
    [
        {
            "system_id": SYSTEM_ID,
            "surface": selected_surface,
            "structure_type": "bare_slab",
            "E_slab_eV": clean_energy_eV,
            **clean_relaxation,
        }
    ]
)

print(f"E_slab (eSEN/OC25) = {clean_energy_eV:.8f} eV")
display(clean_relaxation_table)


E_slab (eSEN/OC25) = -239.05476212 eV


,system_id,surface,structure_type,E_slab_eV,status,error,elapsed_s,n_frames,ml_total_energy_eV,final_fmax_eV_A,converged,trajectory,log
0,mp-10260_111_term0_esen_oc25,mp-10260_111,bare_slab,-239.054762,completed,,2.472518,6,-239.054762,0.044255,True,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...


## 9. Molecular H₂ and H₂O CHE reference energies

After the bare-slab total energy has been obtained, H₂ and H₂O are relaxed
with the **same eSEN/OC25 checkpoint**. Each molecule is placed in a 20 Å cubic
cell with non-periodic boundary conditions and relaxed until

$$
F_{\max}<0.03\ \mathrm{eVA^{-1}}.
$$

These total energies define the water–hydrogen CHE reference terms used in

$$
\Delta E_{\mathrm{ads}}^{\mathrm{CHE}}
=E_{\mathrm{ads+slab}}-E_{\mathrm{slab}}-E_{\mathrm{reference}}.
$$

O₂ is not required for O*, OH*, or OOH* adsorption energies in this reference
scheme. Isolated molecules are outside the principal OC25 interface domain, so
publication values must be recomputed consistently with RPBE-D3.


In [10]:
REFERENCE_DIR = SYSTEM_DIR / "molecular_references"
REFERENCE_DIR.mkdir(parents=True, exist_ok=True)


def prepare_reference_molecule(atoms: Atoms) -> Atoms:
    """Place an isolated molecule in a large cubic cell."""
    prepared = atoms.copy()
    prepared.set_cell(
        [
            REFERENCE_CELL_A,
            REFERENCE_CELL_A,
            REFERENCE_CELL_A,
        ]
    )
    prepared.center()
    prepared.set_pbc(False)
    prepared.set_constraint()
    return prepared


def relax_reference_molecule(
    name: str,
    initial_atoms: Atoms,
) -> dict:
    """Relax one gas-phase CHE reference with eSEN/OC25."""
    trajectory_path = REFERENCE_DIR / f"{name}.traj"
    log_path = REFERENCE_DIR / f"{name}.log"
    xyz_path = REFERENCE_DIR / f"{name}_final.xyz"

    base = {
        "molecule": name,
        "status": "failed",
        "error": "",
        "trajectory": str(trajectory_path),
        "log": str(log_path),
        "final_xyz": str(xyz_path),
        "ml_total_energy_eV": np.nan,
        "final_fmax_eV_A": np.nan,
        "n_frames": 0,
        "converged": False,
    }

    if trajectory_path.exists() and not REFERENCE_OVERWRITE_EXISTING:
        try:
            frames = aseio.read(
                str(trajectory_path),
                index=":",
            )
            if not isinstance(frames, list):
                frames = [frames]

            final = frames[-1]
            energy = float(final.get_potential_energy())
            forces = np.asarray(final.get_forces(), dtype=float)
            fmax = float(
                np.linalg.norm(forces, axis=1).max()
            )

            aseio.write(xyz_path, final)

            return {
                **base,
                "status": "reused",
                "ml_total_energy_eV": energy,
                "final_fmax_eV_A": fmax,
                "n_frames": len(frames),
                "converged": bool(
                    np.isfinite(fmax)
                    and fmax < REFERENCE_FMAX_THRESHOLD
                ),
            }
        except Exception:
            trajectory_path.unlink(missing_ok=True)

    molecule = prepare_reference_molecule(initial_atoms)
    molecule.calc = make_calculator()

    try:
        optimizer = BFGS(
            molecule,
            trajectory=str(trajectory_path),
            logfile=str(log_path),
        )
        optimizer.run(
            fmax=REFERENCE_FMAX_THRESHOLD,
            steps=REFERENCE_MAX_RELAX_STEPS,
        )

        energy = float(molecule.get_potential_energy())
        forces = np.asarray(molecule.get_forces(), dtype=float)
        fmax = float(
            np.linalg.norm(forces, axis=1).max()
        )

        final = molecule.copy()
        final.calc = SinglePointCalculator(
            final,
            energy=energy,
            forces=forces,
        )

        frames = aseio.read(
            str(trajectory_path),
            index=":",
        )
        if not isinstance(frames, list):
            frames = [frames]

        if not frames:
            frames = [final]
        elif (
            np.max(
                np.abs(
                    frames[-1].positions
                    - final.positions
                )
            )
            > 1e-10
        ):
            frames.append(final)
        else:
            frames[-1] = final

        aseio.write(
            str(trajectory_path),
            frames,
            format="traj",
        )
        aseio.write(xyz_path, final)

        return {
            **base,
            "status": "completed",
            "ml_total_energy_eV": energy,
            "final_fmax_eV_A": fmax,
            "n_frames": len(frames),
            "converged": bool(
                np.isfinite(fmax)
                and fmax < REFERENCE_FMAX_THRESHOLD
            ),
        }

    except Exception as exc:
        return {
            **base,
            "error": f"{type(exc).__name__}: {exc}",
        }


molecular_reference_rows = [
    relax_reference_molecule(
        "H2",
        build_h2(),
    ),
    relax_reference_molecule(
        "H2O",
        build_h2o(),
    ),
]

molecular_reference_energies = pd.DataFrame(
    molecular_reference_rows
)

display(molecular_reference_energies)

for row in molecular_reference_rows:
    if row["status"] not in {"completed", "reused"}:
        raise RuntimeError(
            f"{row['molecule']} reference relaxation failed: "
            f"{row['error']}"
        )
    if not row["converged"]:
        raise RuntimeError(
            f"{row['molecule']} did not reach the reference "
            "force threshold."
        )

reference_energy_map = {
    row["molecule"]: float(
        row["ml_total_energy_eV"]
    )
    for row in molecular_reference_rows
}

E_H2_EV = reference_energy_map["H2"]
E_H2O_EV = reference_energy_map["H2O"]

CHE_REFERENCE_ENERGIES_EV = {
    "O": E_H2O_EV - E_H2_EV,
    "OH": E_H2O_EV - 0.5 * E_H2_EV,
    "OOH": 2.0 * E_H2O_EV - 1.5 * E_H2_EV,
}

che_reference_table = pd.DataFrame(
    [
        {
            "adsorbate": adsorbate,
            "reference_expression": {
                "O": "E(H2O) - E(H2)",
                "OH": "E(H2O) - 0.5 E(H2)",
                "OOH": "2 E(H2O) - 1.5 E(H2)",
            }[adsorbate],
            "che_reference_energy_eV": reference_energy,
        }
        for adsorbate, reference_energy
        in CHE_REFERENCE_ENERGIES_EV.items()
    ]
)

print(f"E_eSEN(H2)  = {E_H2_EV:.8f} eV")
print(f"E_eSEN(H2O) = {E_H2O_EV:.8f} eV")
display(che_reference_table)


,molecule,status,error,trajectory,log,final_xyz,ml_total_energy_eV,final_fmax_eV_A,n_frames,converged
0,H2,completed,,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-6.966961,0.002461,2,True
1,H2O,completed,,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-14.155912,0.020792,5,True


E_eSEN(H2)  = -6.96696087 eV
E_eSEN(H2O) = -14.15591156 eV


,adsorbate,reference_expression,che_reference_energy_eV
0,O,E(H2O) - E(H2),-7.188951
1,OH,E(H2O) - 0.5 E(H2),-10.672431
2,OOH,2 E(H2O) - 1.5 E(H2),-17.861382


## 10. Enumerate initial adsorption configurations

Only after the bare-slab and CHE reference energies are available, the notebook
generates the AdsorbML initial structures. For each adsorbate it retains:

- every heuristic AdsorbML placement generated from the selected slab;
- 20 additional placements generated with
  `random_site_heuristic_placement`.

The random-site mode samples the surface triangulation while preserving the
designated binding atom and varying the adsorbate orientation. Heuristic
configurations are not truncated.


In [11]:
placement_rows = []
configuration_records = []

for adsorbate_name in ADSORBATE_NAMES:
    adsorbate = adsorbates[adsorbate_name]

    heuristic = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="heuristic",
    )
    random_sites = AdsorbateSlabConfig(
        slab_object,
        adsorbate,
        mode="random_site_heuristic_placement",
        num_sites=RANDOM_SITES_PER_ADSORBATE,
    )

    groups = [
        ("heuristic", list(heuristic.atoms_list), list(heuristic.metadata_list)),
        ("random_site", list(random_sites.atoms_list), list(random_sites.metadata_list)),
    ]

    running_index = 0
    for placement_kind, atoms_list, metadata_list in groups:
        if len(atoms_list) != len(metadata_list):
            raise RuntimeError("AdsorbML atoms/metadata length mismatch.")

        placement_rows.append(
            {
                "adsorbate": adsorbate_name,
                "placement_kind": placement_kind,
                "n_configurations": len(atoms_list),
            }
        )

        for local_index, (atoms, metadata) in enumerate(zip(atoms_list, metadata_list)):
            prepared = rebuild_atoms(atoms, slab=True)
            ensure_finite_geometry(prepared, f"{adsorbate_name}_{running_index}")

            site = np.asarray(metadata.get("site", [np.nan] * 3), dtype=float)
            configuration_records.append(
                {
                    "config_id": f"{adsorbate_name}_{running_index:03d}",
                    "adsorbate": adsorbate_name,
                    "config_index": running_index,
                    "placement_kind": placement_kind,
                    "placement_local_index": local_index,
                    "initial_site_x_A": float(site[0]),
                    "initial_site_y_A": float(site[1]),
                    "initial_site_z_A": float(site[2]),
                    "sampled_angles": repr(metadata.get("xyz_angles")),
                    "atoms": prepared,
                }
            )
            running_index += 1

placement_summary = (
    pd.DataFrame(placement_rows)
    .pivot(index="adsorbate", columns="placement_kind", values="n_configurations")
    .fillna(0)
    .astype(int)
    .reset_index()
)
placement_summary["total"] = (
    placement_summary.get("heuristic", 0)
    + placement_summary.get("random_site", 0)
)

configuration_table = pd.DataFrame(
    [{k: v for k, v in rec.items() if k != "atoms"} for rec in configuration_records]
)
display(placement_summary)
display(configuration_table.head())


placement_kind,adsorbate,heuristic,random_site,total
0,O,7,20,27
1,OH,7,20,27
2,OOH,7,20,27


,config_id,adsorbate,config_index,placement_kind,placement_local_index,initial_site_x_A,initial_site_y_A,initial_site_z_A,sampled_angles
0,O_000,O,0,heuristic,0,5.291554e+00,4.277113e+00,19.600009,"array([0, 0, 0])"
1,O_001,O,1,heuristic,1,2.116622e+00,1.222032e+00,19.167955,"array([0, 0, 0])"
2,O_002,O,2,heuristic,2,5.644325e+00,4.888129e+00,19.167955,"array([0, 0, 0])"
3,O_003,O,3,heuristic,3,7.049767e-16,4.070185e-16,20.032062,"array([0, 0, 0])"
4,O_004,O,4,heuristic,4,5.291554e+00,5.499145e+00,18.735902,"array([0, 0, 0])"


## 11. Adsorbate–slab relaxation and CHE adsorption energies

Every AdsorbML configuration is relaxed independently. For each readable final
structure, the notebook records the eSEN total energy and immediately evaluates

$$
\Delta E_{\mathrm{ads}}^{\mathrm{CHE}}
=E_{\mathrm{ads+slab}}-E_{\mathrm{slab}}-E_{\mathrm{reference}}.
$$

No total-energy column is reinterpreted as an adsorption energy. The raw total
energy and every term entering the subtraction are retained in the exported
CSV tables.


In [12]:
relaxation_rows = []

for record in tqdm(configuration_records, desc="eSEN/OC25 relaxations"):
    result = relax_structure(
        record["atoms"],
        TRAJECTORY_DIR / record["adsorbate"] / f"{record['config_id']}.traj",
        LOG_DIR / record["adsorbate"] / f"{record['config_id']}.log",
    )
    relaxation_rows.append(
        {
            "system_id": SYSTEM_ID,
            "config_id": record["config_id"],
            "adsorbate": record["adsorbate"],
            "config_index": int(record["config_index"]),
            "placement_kind": record["placement_kind"],
            **result,
        }
    )

relaxation_status = pd.DataFrame(relaxation_rows)

# eSEN predicts total energies. The adsorption energy is derived only after
# subtracting the independently relaxed slab and the CHE reference energy.
relaxation_status["E_slab_eV"] = clean_energy_eV
relaxation_status["E_adslab_eV"] = pd.to_numeric(
    relaxation_status["ml_total_energy_eV"],
    errors="coerce",
)
relaxation_status["adslab_minus_slab_eV"] = (
    relaxation_status["E_adslab_eV"]
    - relaxation_status["E_slab_eV"]
)
relaxation_status["che_reference_energy_eV"] = (
    relaxation_status["adsorbate"]
    .map(CHE_REFERENCE_ENERGIES_EV)
    .astype(float)
)
relaxation_status["eSEN_pred_ads_energy_eV"] = (
    relaxation_status["E_adslab_eV"]
    - relaxation_status["E_slab_eV"]
    - relaxation_status["che_reference_energy_eV"]
)

energy_columns = [
    "adsorbate",
    "config_id",
    "status",
    "converged",
    "E_adslab_eV",
    "E_slab_eV",
    "che_reference_energy_eV",
    "eSEN_pred_ads_energy_eV",
]

print("Adsorption-energy convention: E_ads+slab - E_slab - CHE reference")
display(relaxation_status[energy_columns])
display(
    relaxation_status.groupby(
        ["adsorbate", "status", "converged"],
        dropna=False,
    )
    .size()
    .rename("count")
    .reset_index()
)


eSEN/OC25 relaxations:   0%|          | 0/81 [00:00<?, ?it/s]

Adsorption-energy convention: E_ads+slab - E_slab - CHE reference


,adsorbate,config_id,status,converged,E_adslab_eV,E_slab_eV,che_reference_energy_eV,eSEN_pred_ads_energy_eV
0,O,O_000,completed,True,-244.636139,-239.054762,-7.188951,1.607574
1,O,O_001,completed,True,-244.110370,-239.054762,-7.188951,2.133343
2,O,O_002,completed,True,-244.634836,-239.054762,-7.188951,1.608876
3,O,O_003,completed,True,-244.103869,-239.054762,-7.188951,2.139844
4,O,O_004,completed,True,-244.496907,-239.054762,-7.188951,1.746806
...,...,...,...,...,...,...,...,...
76,OOH,OOH_022,completed,True,-253.027839,-239.054762,-17.861382,3.888305
77,OOH,OOH_023,completed,True,-253.022371,-239.054762,-17.861382,3.893773
78,OOH,OOH_024,completed,True,-253.027730,-239.054762,-17.861382,3.888414
79,OOH,OOH_025,completed,True,-252.893475,-239.054762,-17.861382,4.022669


,adsorbate,status,converged,count
0,O,completed,True,27
1,OH,completed,True,27
2,OOH,completed,True,27


## 12. Trajectory validation

A relaxed adsorbate structure is accepted only when:

- the final positions, lattice vectors, energy, and forces are finite;
- the requested force threshold is reached;
- the adsorbate remains chemically intact;
- the adsorbate is neither desorbed nor intercalated; and
- the surface does not trigger the FAIRChem reconstruction check.

Every readable final frame is exported as a POSCAR, including rejected
configurations, to support manual inspection and provenance tracking.


In [13]:
def adsorbate_indices_from_tags(atoms: Atoms, adsorbate_name: str) -> np.ndarray:
    tags = np.asarray(atoms.get_tags(), dtype=int)
    tagged = np.flatnonzero(tags == 2)
    expected = ADSORBATE_SIZES[adsorbate_name]
    if len(tagged) == expected:
        return tagged

    fallback = np.arange(len(atoms) - expected, len(atoms), dtype=int)
    tags[fallback] = 2
    atoms.set_tags(tags)
    return fallback

def safe_anomaly_flags(frames: list[Atoms], adsorbate_name: str) -> dict:
    initial = frames[0].copy()
    final = frames[-1].copy()

    if (
        not np.isfinite(initial.positions).all()
        or not np.isfinite(final.positions).all()
        or not np.isfinite(final.cell.array).all()
    ):
        return {
            "adsorbate_dissociated": True,
            "adsorbate_desorbed": True,
            "surface_changed": True,
            "adsorbate_intercalated": True,
            "anomaly_check_error": "non_finite_geometry",
        }

    adsorbate_indices_from_tags(initial, adsorbate_name)
    adsorbate_indices_from_tags(final, adsorbate_name)

    try:
        detector = DetectTrajAnomaly(
            initial,
            final,
            np.asarray(initial.get_tags(), dtype=int),
        )
        return {
            "adsorbate_dissociated": bool(detector.is_adsorbate_dissociated()),
            "adsorbate_desorbed": bool(detector.is_adsorbate_desorbed()),
            "surface_changed": bool(detector.has_surface_changed()),
            "adsorbate_intercalated": bool(detector.is_adsorbate_intercalated()),
            "anomaly_check_error": "",
        }
    except Exception as exc:
        return {
            "adsorbate_dissociated": True,
            "adsorbate_desorbed": True,
            "surface_changed": True,
            "adsorbate_intercalated": True,
            "anomaly_check_error": f"{type(exc).__name__}: {exc}",
        }

def write_poscar(atoms: Atoms, directory: Path) -> Path:
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / "POSCAR"
    aseio.write(path, atoms, format="vasp", direct=True, vasp5=True, sort=True)
    return path

validation_rows = []

for row in relaxation_status.itertuples(index=False):
    record = {
        "system_id": row.system_id,
        "config_id": row.config_id,
        "adsorbate": row.adsorbate,
        "config_index": int(row.config_index),
        "placement_kind": row.placement_kind,
        "trajectory": row.trajectory,
        "relaxation_status": row.status,
        "validation_status": "rejected",
        "rejection_reason": "",
        "ml_total_energy_eV": row.ml_total_energy_eV,
        "E_adslab_eV": row.E_adslab_eV,
        "E_slab_eV": row.E_slab_eV,
        "adslab_minus_slab_eV": row.adslab_minus_slab_eV,
        "che_reference_energy_eV": row.che_reference_energy_eV,
        "eSEN_pred_ads_energy_eV": row.eSEN_pred_ads_energy_eV,
        "final_fmax_eV_A": row.final_fmax_eV_A,
        "n_frames": int(row.n_frames),
        "final_poscar": "",
        "adsorbate_dissociated": False,
        "adsorbate_desorbed": False,
        "surface_changed": False,
        "adsorbate_intercalated": False,
        "anomaly_check_error": "",
    }

    if row.status not in {"completed", "reused"}:
        record["rejection_reason"] = row.error or "relaxation_failed"
        validation_rows.append(record)
        continue

    try:
        frames = read_frames(Path(row.trajectory))
        final = frames[-1]
        record["final_poscar"] = str(
            write_poscar(
                final,
                POSCAR_DIR / "all_final_frames" / row.adsorbate / row.config_id,
            )
        )

        flags = safe_anomaly_flags(frames, row.adsorbate)
        record.update(flags)
        active = [
            key for key in (
                "adsorbate_dissociated",
                "adsorbate_desorbed",
                "surface_changed",
                "adsorbate_intercalated",
            )
            if flags[key]
        ]

        if not np.isfinite(row.ml_total_energy_eV):
            record["rejection_reason"] = "non_finite_energy"
        elif not np.isfinite(row.final_fmax_eV_A):
            record["rejection_reason"] = "non_finite_force"
        elif row.final_fmax_eV_A >= FMAX_THRESHOLD:
            record["rejection_reason"] = (
                f"force_above_threshold:{row.final_fmax_eV_A:.6f}"
            )
        elif active:
            record["rejection_reason"] = ";".join(active)
        else:
            record["validation_status"] = "accepted"
    except Exception as exc:
        record["rejection_reason"] = (
            f"validation_failed:{type(exc).__name__}:{exc}"
        )

    validation_rows.append(record)

trajectory_validation = pd.DataFrame(validation_rows)
display(trajectory_validation)
display(
    trajectory_validation.groupby(["adsorbate", "validation_status"])
    .size()
    .rename("count")
    .reset_index()
)


,system_id,config_id,adsorbate,config_index,placement_kind,trajectory,relaxation_status,validation_status,rejection_reason,ml_total_energy_eV,...,che_reference_energy_eV,eSEN_pred_ads_energy_eV,final_fmax_eV_A,n_frames,final_poscar,adsorbate_dissociated,adsorbate_desorbed,surface_changed,adsorbate_intercalated,anomaly_check_error
0,mp-10260_111_term0_esen_oc25,O_000,O,0,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-244.636139,...,-7.188951,1.607574,0.046254,35,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
1,mp-10260_111_term0_esen_oc25,O_001,O,1,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-244.110370,...,-7.188951,2.133343,0.035834,16,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
2,mp-10260_111_term0_esen_oc25,O_002,O,2,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-244.634836,...,-7.188951,1.608876,0.046841,36,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
3,mp-10260_111_term0_esen_oc25,O_003,O,3,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-244.103869,...,-7.188951,2.139844,0.024478,21,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
4,mp-10260_111_term0_esen_oc25,O_004,O,4,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-244.496907,...,-7.188951,1.746806,0.040276,27,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,mp-10260_111_term0_esen_oc25,OOH_022,OOH,22,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-253.027839,...,-17.861382,3.888305,0.044609,134,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
77,mp-10260_111_term0_esen_oc25,OOH_023,OOH,23,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-253.022371,...,-17.861382,3.893773,0.049410,80,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
78,mp-10260_111_term0_esen_oc25,OOH_024,OOH,24,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-253.027730,...,-17.861382,3.888414,0.045036,69,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,
79,mp-10260_111_term0_esen_oc25,OOH_025,OOH,25,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,completed,accepted,,-252.893475,...,-17.861382,4.022669,0.047009,50,/content/stageII_esen_oc25_results/mp-10260_11...,False,False,False,False,


,adsorbate,validation_status,count
0,O,accepted,27
1,OH,accepted,27
2,OOH,accepted,24
3,OOH,rejected,3


## 13. Adsorption-site and uniqueness analysis

The binding oxygen is identified from the tag-2 adsorbate atoms as the
oxygen closest to the tagged surface. Its in-plane projection is compared
with nearby surface atoms to classify the final geometry as top, bridge,
threefold hollow, fourfold, or a distance-based fallback.

The analysis reports three complementary levels:

1. **Accepted configurations** — all valid final trajectories.
2. **Unique site instances** — configurations grouped by the explicit set
   of neighboring surface-atom indices.
3. **Site families** — configurations grouped by coordination and the
   chemical identities of the neighboring atoms, for example
   `bridge__Ni-Pt`.

The lowest CHE electronic adsorption energy is retained within each
fixed-composition group, followed by one global minimum for each adsorbate.
Because the slab and reference terms are constant within one adsorbate, this
selection is numerically identical to ranking by adsorbate–slab total energy.


In [14]:
LOCAL_XY_RADIUS = 4.8
SURFACE_LAYER_TOL = 1.8
TOP_XY_TOL = 0.45
BRIDGE_PROJECTION_TOL = 0.65
MAX_BRIDGE_LENGTH = 4.5
MAX_HOLLOW_EDGE = 4.8
BOND_BUFFER = 0.65
MAX_BOND_DISTANCE = 3.1
DISTANCE_TIE_TOL = 0.22

def mic_vectors_xy(
    atoms: Atoms,
    anchor_position: np.ndarray,
    positions: np.ndarray,
) -> np.ndarray:
    cell = np.asarray(atoms.cell.array, dtype=float)
    inverse_cell = np.linalg.inv(cell)
    anchor_fractional = anchor_position @ inverse_cell
    fractional = positions @ inverse_cell
    delta = fractional - anchor_fractional
    delta[:, 0] -= np.round(delta[:, 0])
    delta[:, 1] -= np.round(delta[:, 1])
    return delta @ cell

def point_segment_distance_2d(
    point: np.ndarray,
    start: np.ndarray,
    end: np.ndarray,
) -> tuple[float, float]:
    segment = end - start
    denominator = float(np.dot(segment, segment))
    if denominator < 1e-12:
        return float(np.linalg.norm(point - start)), 0.0
    parameter = float(np.dot(point - start, segment) / denominator)
    clipped = float(np.clip(parameter, 0.0, 1.0))
    closest = start + clipped * segment
    return float(np.linalg.norm(point - closest)), clipped

def point_in_triangle_2d(
    point: np.ndarray,
    a: np.ndarray,
    b: np.ndarray,
    c: np.ndarray,
    epsilon: float = 1e-8,
) -> bool:
    v0 = c - a
    v1 = b - a
    v2 = point - a
    denominator = v0[0] * v1[1] - v1[0] * v0[1]
    if abs(denominator) < 1e-12:
        return False
    u = (v2[0] * v1[1] - v1[0] * v2[1]) / denominator
    v = (v0[0] * v2[1] - v2[0] * v0[1]) / denominator
    return bool(u >= -epsilon and v >= -epsilon and u + v <= 1.0 + epsilon)

def site_type_from_coordination(coordination: int) -> str:
    return {
        1: "top",
        2: "bridge",
        3: "3-fold",
        4: "4-fold",
    }.get(coordination, "unknown" if coordination <= 0 else f"{coordination}-fold")

def format_site_result(selected: list[dict], reason: str) -> dict:
    selected = sorted(selected, key=lambda item: item["index"])
    indices = [int(item["index"]) for item in selected]
    symbols = [item["symbol"] for item in selected]
    distances = [float(item["r3d"]) for item in selected]

    site_type = site_type_from_coordination(len(selected))
    neighbor_composition = "-".join(sorted(symbols))
    distance_fingerprint = "-".join(f"{x:.2f}" for x in sorted(distances))
    site_instance_key = f"{site_type}__idx_" + "-".join(map(str, indices))
    site_family = f"{site_type}__{neighbor_composition}"

    return {
        "site_type": site_type,
        "coordination": len(selected),
        "neighbor_composition": neighbor_composition,
        "neighbor_indices": "-".join(map(str, indices)),
        "neighbor_distances_A": ";".join(f"{x:.4f}" for x in distances),
        "site_instance_key": site_instance_key,
        "site_family": site_family,
        "local_site_descriptor": f"{site_family}__d_{distance_fingerprint}",
        "site_assignment_reason": reason,
    }

def choose_binding_oxygen(
    atoms: Atoms,
    adsorbate_indices: np.ndarray,
    surface_indices: np.ndarray,
) -> int:
    symbols = atoms.get_chemical_symbols()
    oxygen_indices = [
        int(i) for i in adsorbate_indices if symbols[i] == "O"
    ]
    if not oxygen_indices:
        raise RuntimeError("No oxygen atom found in the tagged adsorbate.")
    if len(oxygen_indices) == 1:
        return oxygen_indices[0]

    positions = atoms.get_positions()
    surface_positions = positions[surface_indices]
    return min(
        oxygen_indices,
        key=lambda i: float(
            np.linalg.norm(
                mic_vectors_xy(atoms, positions[i], surface_positions),
                axis=1,
            ).min()
        ),
    )

def detect_adsorption_site(atoms: Atoms, adsorbate_name: str) -> dict:
    atoms = atoms.copy()
    adsorbate_indices = adsorbate_indices_from_tags(atoms, adsorbate_name)
    tags = np.asarray(atoms.get_tags(), dtype=int)
    surface_indices = np.flatnonzero(tags == 1)
    if len(surface_indices) == 0:
        surface_indices = np.flatnonzero(tags != 2)

    positions = atoms.get_positions()
    symbols = atoms.get_chemical_symbols()
    binding_oxygen = choose_binding_oxygen(
        atoms,
        adsorbate_indices,
        surface_indices,
    )
    anchor = positions[binding_oxygen]
    surface_positions = positions[surface_indices]
    vectors = mic_vectors_xy(atoms, anchor, surface_positions)
    r3d = np.linalg.norm(vectors, axis=1)
    rxy = np.linalg.norm(vectors[:, :2], axis=1)

    surface_z = surface_positions[:, 2]
    local_mask = (
        (surface_z >= float(surface_z.max()) - SURFACE_LAYER_TOL)
        & (rxy <= LOCAL_XY_RADIUS)
    )
    local_ids = np.flatnonzero(local_mask)
    if len(local_ids) == 0:
        local_ids = np.argsort(r3d)[:12]
    local_ids = sorted(local_ids, key=lambda i: (rxy[i], r3d[i]))[:16]

    candidates = [
        {
            "index": int(surface_indices[i]),
            "symbol": symbols[int(surface_indices[i])],
            "xy": vectors[i, :2],
            "rxy": float(rxy[i]),
            "r3d": float(r3d[i]),
        }
        for i in local_ids
    ]

    origin = np.zeros(2)
    nearest = min(candidates, key=lambda item: item["rxy"])

    if nearest["rxy"] <= TOP_XY_TOL:
        result = format_site_result([nearest], "projection_top")
        result["binding_oxygen_index"] = binding_oxygen
        return result

    bridge_options = []
    for atom_a, atom_b in itertools.combinations(candidates, 2):
        pair_length = float(np.linalg.norm(atom_a["xy"] - atom_b["xy"]))
        if pair_length > MAX_BRIDGE_LENGTH:
            continue
        distance, parameter = point_segment_distance_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
        )
        if distance <= BRIDGE_PROJECTION_TOL and 0.0 <= parameter <= 1.0:
            score = (
                distance
                + 0.20 * abs(parameter - 0.5)
                + 0.03 * (atom_a["r3d"] + atom_b["r3d"])
            )
            bridge_options.append((score, atom_a, atom_b))

    if bridge_options:
        _, atom_a, atom_b = min(bridge_options, key=lambda item: item[0])
        result = format_site_result([atom_a, atom_b], "projection_bridge")
        result["binding_oxygen_index"] = binding_oxygen
        return result

    hollow_options = []
    for atom_a, atom_b, atom_c in itertools.combinations(candidates, 3):
        edges = [
            float(np.linalg.norm(atom_a["xy"] - atom_b["xy"])),
            float(np.linalg.norm(atom_a["xy"] - atom_c["xy"])),
            float(np.linalg.norm(atom_b["xy"] - atom_c["xy"])),
        ]
        if max(edges) > MAX_HOLLOW_EDGE:
            continue
        if point_in_triangle_2d(
            origin,
            atom_a["xy"],
            atom_b["xy"],
            atom_c["xy"],
        ):
            centroid = (atom_a["xy"] + atom_b["xy"] + atom_c["xy"]) / 3.0
            score = float(np.linalg.norm(centroid)) + 0.02 * (
                atom_a["r3d"] + atom_b["r3d"] + atom_c["r3d"]
            )
            hollow_options.append((score, atom_a, atom_b, atom_c))

    if hollow_options:
        _, atom_a, atom_b, atom_c = min(
            hollow_options,
            key=lambda item: item[0],
        )
        result = format_site_result(
            [atom_a, atom_b, atom_c],
            "projection_hollow",
        )
        result["binding_oxygen_index"] = binding_oxygen
        return result

    oxygen_radius = covalent_radii[atomic_numbers["O"]]
    bonded = []
    for candidate in candidates:
        surface_radius = covalent_radii[atomic_numbers[candidate["symbol"]]]
        expected = oxygen_radius + surface_radius
        normalized = candidate["r3d"] / expected if expected > 0 else candidate["r3d"]
        if (
            candidate["r3d"] <= expected + BOND_BUFFER
            or candidate["r3d"] <= MAX_BOND_DISTANCE
        ):
            bonded.append((normalized, candidate))

    if bonded:
        minimum_score = min(item[0] for item in bonded)
        selected = [
            item[1]
            for item in bonded
            if item[0] <= minimum_score + DISTANCE_TIE_TOL
        ]
        selected = sorted(selected, key=lambda item: item["r3d"])[:4]
        result = format_site_result(selected, "bond_distance_fallback")
    else:
        result = format_site_result([nearest], "nearest_projection_fallback")

    result["binding_oxygen_index"] = binding_oxygen
    return result


In [15]:
accepted = trajectory_validation[
    trajectory_validation["validation_status"] == "accepted"
].copy()

site_rows = []
for row in accepted.itertuples(index=False):
    final = aseio.read(row.trajectory, index=-1)
    site_rows.append(
        {
            "system_id": row.system_id,
            "config_id": row.config_id,
            "adsorbate": row.adsorbate,
            "config_index": int(row.config_index),
            "placement_kind": row.placement_kind,
            "trajectory": row.trajectory,
            "final_poscar": row.final_poscar,
            "ml_total_energy_eV": float(row.ml_total_energy_eV),
            "E_adslab_eV": float(row.E_adslab_eV),
            "E_slab_eV": float(row.E_slab_eV),
            "adslab_minus_slab_eV": float(row.adslab_minus_slab_eV),
            "che_reference_energy_eV": float(row.che_reference_energy_eV),
            "eSEN_pred_ads_energy_eV": float(row.eSEN_pred_ads_energy_eV),
            "final_fmax_eV_A": float(row.final_fmax_eV_A),
            **detect_adsorption_site(final, row.adsorbate),
        }
    )

all_adsorption_sites = pd.DataFrame(site_rows)

if all_adsorption_sites.empty:
    unique_site_minima = pd.DataFrame()
    site_family_minima = pd.DataFrame()
    global_minima = pd.DataFrame()
else:
    all_adsorption_sites["delta_from_adsorbate_min_eV"] = (
        all_adsorption_sites["eSEN_pred_ads_energy_eV"]
        - all_adsorption_sites.groupby("adsorbate")["eSEN_pred_ads_energy_eV"]
        .transform("min")
    )

    unique_site_minima = (
        all_adsorption_sites
        .sort_values("eSEN_pred_ads_energy_eV")
        .groupby(["system_id", "adsorbate", "site_instance_key"], as_index=False)
        .first()
        .sort_values(["adsorbate", "eSEN_pred_ads_energy_eV"])
        .reset_index(drop=True)
    )

    site_family_minima = (
        all_adsorption_sites
        .sort_values("eSEN_pred_ads_energy_eV")
        .groupby(["system_id", "adsorbate", "site_family"], as_index=False)
        .first()
        .sort_values(["adsorbate", "eSEN_pred_ads_energy_eV"])
        .reset_index(drop=True)
    )

    global_minima = (
        all_adsorption_sites
        .sort_values("eSEN_pred_ads_energy_eV")
        .groupby(["system_id", "adsorbate"], as_index=False)
        .first()
        .sort_values("adsorbate")
        .reset_index(drop=True)
    )

print("All accepted configurations")
display(all_adsorption_sites)
print("Unique site-instance minima")
display(unique_site_minima)
print("Site-family minima")
display(site_family_minima)
print("Global minimum per adsorbate")
display(global_minima)


All accepted configurations


,system_id,config_id,adsorbate,config_index,placement_kind,trajectory,final_poscar,ml_total_energy_eV,E_adslab_eV,E_slab_eV,...,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_esen_oc25,O_000,O,0,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.636139,-244.636139,-239.054762,...,2,Ni-Sb,16-45,1.8903;1.9867,bridge__idx_16-45,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.003340
1,mp-10260_111_term0_esen_oc25,O_001,O,1,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.110370,-244.110370,-239.054762,...,1,Ni,4,1.8796,top__idx_4,top__Ni,top__Ni__d_1.88,projection_top,48,0.529110
2,mp-10260_111_term0_esen_oc25,O_002,O,2,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.634836,-244.634836,-239.054762,...,2,Ni-Sb,16-45,1.8922;1.9901,bridge__idx_16-45,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.004643
3,mp-10260_111_term0_esen_oc25,O_003,O,3,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.103869,-244.103869,-239.054762,...,1,Sb,9,1.8682,top__idx_9,top__Sb,top__Sb__d_1.87,projection_top,48,0.535611
4,mp-10260_111_term0_esen_oc25,O_004,O,4,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.496907,-244.496907,-239.054762,...,2,Ni-Ni,14-16,2.9272;1.9192,bridge__idx_14-16,bridge__Ni-Ni,bridge__Ni-Ni__d_1.92-2.93,projection_bridge,48,0.142573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,mp-10260_111_term0_esen_oc25,OOH_022,OOH,22,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-253.027839,-253.027839,-239.054762,...,1,Sb,33,2.1094,top__idx_33,top__Sb,top__Sb__d_2.11,projection_top,48,0.000000
74,mp-10260_111_term0_esen_oc25,OOH_023,OOH,23,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-253.022371,-253.022371,-239.054762,...,1,Sb,45,2.1086,top__idx_45,top__Sb,top__Sb__d_2.11,projection_top,48,0.005468
75,mp-10260_111_term0_esen_oc25,OOH_024,OOH,24,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-253.027730,-253.027730,-239.054762,...,1,Sb,33,2.1002,top__idx_33,top__Sb,top__Sb__d_2.10,projection_top,48,0.000109
76,mp-10260_111_term0_esen_oc25,OOH_025,OOH,25,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-252.893475,-252.893475,-239.054762,...,2,Ni-Ni,4-16,3.8146;2.0495,bridge__idx_4-16,bridge__Ni-Ni,bridge__Ni-Ni__d_2.05-3.81,projection_bridge,48,0.134364


Unique site-instance minima


,system_id,adsorbate,site_instance_key,config_id,config_index,placement_kind,trajectory,final_poscar,ml_total_energy_eV,E_adslab_eV,...,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_esen_oc25,O,bridge__idx_33-40,O_012,12,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.639480,-244.639480,...,bridge,2,Ni-Sb,33-40,1.9908;1.8921,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.000000
1,mp-10260_111_term0_esen_oc25,O,bridge__idx_16-45,O_024,24,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.638994,-244.638994,...,bridge,2,Ni-Sb,16-45,1.8867;1.9908,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.000486
2,mp-10260_111_term0_esen_oc25,O,bridge__idx_9-28,O_021,21,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.638626,-244.638626,...,bridge,2,Ni-Sb,9-28,1.9921;1.8926,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.000853
3,mp-10260_111_term0_esen_oc25,O,bridge__idx_16-21,O_008,8,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.638021,-244.638021,...,bridge,2,Ni-Sb,16-21,1.8912;1.9913,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.001458
4,mp-10260_111_term0_esen_oc25,O,bridge__idx_21-40,O_009,9,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.637339,-244.637339,...,bridge,2,Ni-Sb,21-40,1.9888;1.8888,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.002140
5,mp-10260_111_term0_esen_oc25,O,bridge__idx_4-21,O_010,10,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.636955,-244.636955,...,bridge,2,Ni-Sb,4-21,1.8909;1.9903,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.002525
6,mp-10260_111_term0_esen_oc25,O,bridge__idx_40-45,O_011,11,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.635934,-244.635934,...,bridge,2,Ni-Sb,40-45,1.8916;1.9893,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.003546
7,mp-10260_111_term0_esen_oc25,O,bridge__idx_28-33,O_018,18,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.634924,-244.634924,...,bridge,2,Ni-Sb,28-33,1.8917;1.9874,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.004556
8,mp-10260_111_term0_esen_oc25,O,bridge__idx_4-9,O_020,20,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.634592,-244.634592,...,bridge,2,Ni-Sb,4-9,1.8915;1.9904,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.004887
9,mp-10260_111_term0_esen_oc25,O,bridge__idx_14-16,O_004,4,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.496907,-244.496907,...,bridge,2,Ni-Ni,14-16,2.9272;1.9192,bridge__Ni-Ni,bridge__Ni-Ni__d_1.92-2.93,projection_bridge,48,0.142573


Site-family minima


,system_id,adsorbate,site_family,config_id,config_index,placement_kind,trajectory,final_poscar,ml_total_energy_eV,E_adslab_eV,...,site_type,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_esen_oc25,O,bridge__Ni-Sb,O_012,12,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.639480,-244.639480,...,bridge,2,Ni-Sb,33-40,1.9908;1.8921,bridge__idx_33-40,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.000000
1,mp-10260_111_term0_esen_oc25,O,bridge__Ni-Ni,O_004,4,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.496907,-244.496907,...,bridge,2,Ni-Ni,14-16,2.9272;1.9192,bridge__idx_14-16,bridge__Ni-Ni__d_1.92-2.93,projection_bridge,48,0.142573
2,mp-10260_111_term0_esen_oc25,O,top__Ni,O_001,1,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.110370,-244.110370,...,top,1,Ni,4,1.8796,top__idx_4,top__Ni__d_1.88,projection_top,48,0.529110
3,mp-10260_111_term0_esen_oc25,O,top__Sb,O_003,3,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.103869,-244.103869,...,top,1,Sb,9,1.8682,top__idx_9,top__Sb__d_1.87,projection_top,48,0.535611
4,mp-10260_111_term0_esen_oc25,O,3-fold__Ni-Ni-Ni,O_006,6,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-243.914137,-243.914137,...,3-fold,3,Ni-Ni-Ni,4-16-28,2.4744;2.4743;2.4744,3-fold__idx_4-16-28,3-fold__Ni-Ni-Ni__d_2.47-2.47-2.47,projection_hollow,48,0.725343
5,mp-10260_111_term0_esen_oc25,OH,top__Sb,OH_001,1,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-249.046599,-249.046599,...,top,1,Sb,9,2.0317,top__idx_9,top__Sb__d_2.03,projection_top,48,0.000000
6,mp-10260_111_term0_esen_oc25,OH,bridge__Ni-Sb,OH_015,15,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-248.904692,-248.904692,...,bridge,2,Ni-Sb,28-45,2.0915;2.2005,bridge__idx_28-45,bridge__Ni-Sb__d_2.09-2.20,projection_bridge,48,0.141907
7,mp-10260_111_term0_esen_oc25,OH,bridge__Ni-Ni,OH_008,8,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-248.890453,-248.890453,...,bridge,2,Ni-Ni,16-40,3.8876;2.0919,bridge__idx_16-40,bridge__Ni-Ni__d_2.09-3.89,projection_bridge,48,0.156146
8,mp-10260_111_term0_esen_oc25,OOH,top__Sb,OOH_022,22,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-253.027839,-253.027839,...,top,1,Sb,33,2.1094,top__idx_33,top__Sb__d_2.11,projection_top,48,0.000000
9,mp-10260_111_term0_esen_oc25,OOH,bridge__Sb-Sb,OOH_014,14,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-252.991209,-252.991209,...,bridge,2,Sb-Sb,21-45,2.1056;4.1161,bridge__idx_21-45,bridge__Sb-Sb__d_2.11-4.12,projection_bridge,48,0.036630


Global minimum per adsorbate


,system_id,adsorbate,config_id,config_index,placement_kind,trajectory,final_poscar,ml_total_energy_eV,E_adslab_eV,E_slab_eV,...,coordination,neighbor_composition,neighbor_indices,neighbor_distances_A,site_instance_key,site_family,local_site_descriptor,site_assignment_reason,binding_oxygen_index,delta_from_adsorbate_min_eV
0,mp-10260_111_term0_esen_oc25,O,O_012,12,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-244.639480,-244.639480,-239.054762,...,2,Ni-Sb,33-40,1.9908;1.8921,bridge__idx_33-40,bridge__Ni-Sb,bridge__Ni-Sb__d_1.89-1.99,projection_bridge,48,0.0
1,mp-10260_111_term0_esen_oc25,OH,OH_001,1,heuristic,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-249.046599,-249.046599,-239.054762,...,1,Sb,9,2.0317,top__idx_9,top__Sb,top__Sb__d_2.03,projection_top,48,0.0
2,mp-10260_111_term0_esen_oc25,OOH,OOH_022,22,random_site,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,-253.027839,-253.027839,-239.054762,...,1,Sb,33,2.1094,top__idx_33,top__Sb,top__Sb__d_2.11,projection_top,48,0.0


## 14. Verify and summarize eSEN adsorption energies

The adsorption-energy column is created directly after every adsorbate
relaxation. This section verifies that every accepted and selected structure
satisfies

$$
E_{\mathrm{ads}}^{\mathrm{eSEN/CHE}}
=E_{\mathrm{ads+slab}}-E_{\mathrm{slab}}-E_{\mathrm{reference}}
$$

to numerical precision, then reports the global O*, OH*, and OOH* minima.

This notebook intentionally stops at $E_{\mathrm{ads}}$. ZPE, entropy,
solvation, potential-dependent step free energies, limiting potential, and
overpotential are evaluated only after the RPBE-D3 and VaspGibbs/VASPsol stage.


In [16]:
def verify_adsorption_energy(
    table: pd.DataFrame,
    *,
    table_name: str,
    atol: float = 1e-8,
) -> pd.DataFrame:
    """Verify E_ads+slab - E_slab - E_reference for one result table."""
    if table.empty:
        return table.copy()

    checked = table.copy()
    required = {
        "E_adslab_eV",
        "E_slab_eV",
        "che_reference_energy_eV",
        "eSEN_pred_ads_energy_eV",
    }
    missing = required - set(checked.columns)
    if missing:
        raise KeyError(
            f"{table_name} is missing energy columns: "
            + ", ".join(sorted(missing))
        )

    recomputed = (
        checked["E_adslab_eV"].astype(float)
        - checked["E_slab_eV"].astype(float)
        - checked["che_reference_energy_eV"].astype(float)
    )
    reported = checked["eSEN_pred_ads_energy_eV"].astype(float)
    finite = np.isfinite(recomputed) & np.isfinite(reported)

    if finite.any() and not np.allclose(
        reported.loc[finite],
        recomputed.loc[finite],
        rtol=0.0,
        atol=atol,
    ):
        maximum_error = float(
            np.max(np.abs(reported.loc[finite] - recomputed.loc[finite]))
        )
        raise RuntimeError(
            f"Adsorption-energy check failed for {table_name}; "
            f"maximum mismatch = {maximum_error:.3e} eV."
        )

    checked["ads_energy_formula_check_eV"] = reported - recomputed
    return checked


all_adsorption_sites = verify_adsorption_energy(
    all_adsorption_sites,
    table_name="all_adsorption_sites",
)
unique_site_minima = verify_adsorption_energy(
    unique_site_minima,
    table_name="unique_site_minima",
)
site_family_minima = verify_adsorption_energy(
    site_family_minima,
    table_name="site_family_minima",
)
global_minima = verify_adsorption_energy(
    global_minima,
    table_name="global_minima",
)

required_adsorbates = set(ADSORBATE_NAMES)
available_adsorbates = set(
    global_minima.get("adsorbate", pd.Series(dtype=str)).tolist()
)
missing_adsorbates = required_adsorbates - available_adsorbates

if missing_adsorbates:
    raise RuntimeError(
        "The analysis requires accepted global minima for O, OH, and OOH. "
        "Missing: " + ", ".join(sorted(missing_adsorbates))
    )

global_adsorption_energies = (
    global_minima
    .set_index("adsorbate")
    .loc[list(ADSORBATE_NAMES)]
    .reset_index()
)

energy_summary_columns = [
    "adsorbate",
    "config_id",
    "site_family",
    "E_adslab_eV",
    "E_slab_eV",
    "che_reference_energy_eV",
    "eSEN_pred_ads_energy_eV",
    "ads_energy_formula_check_eV",
]

print("Global eSEN/OC25 electronic adsorption energies")
print("Formula: E_ads+slab - E_slab - E_reference")
display(global_adsorption_energies[energy_summary_columns])

max_formula_error = float(
    np.nanmax(
        np.abs(
            global_adsorption_energies[
                "ads_energy_formula_check_eV"
            ].to_numpy(dtype=float)
        )
    )
)
print(f"Maximum adsorption-energy formula mismatch: {max_formula_error:.3e} eV")


Global eSEN/OC25 electronic adsorption energies
Formula: E_ads+slab - E_slab - E_reference


,adsorbate,config_id,site_family,E_adslab_eV,E_slab_eV,che_reference_energy_eV,eSEN_pred_ads_energy_eV,ads_energy_formula_check_eV
0,O,O_012,bridge__Ni-Sb,-244.639480,-239.054762,-7.188951,1.604233,0.0
1,OH,OH_001,top__Sb,-249.046599,-239.054762,-10.672431,0.680594,0.0
2,OOH,OOH_022,top__Sb,-253.027839,-239.054762,-17.861382,3.888305,0.0


Maximum adsorption-energy formula mismatch: 0.000e+00 eV


## 15. Interactive structural inspection

The dropdown contains the clean eSEN-relaxed slab, the global minimum for each
adsorbate, and the minimum representative of every unique adsorption site.
Adsorbate entries display `eSEN_pred_ads_energy_eV`, not the raw total energy.

For easier inspection across periodic boundaries, every selected structure is
shown as a 2 × 2 × 1 supercell. This replication is used only for visualization
and does not change any stored trajectory or calculated energy.


In [17]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass


def atoms_to_cif_string(atoms: Atoms) -> str:
    buffer = io.BytesIO()
    aseio.write(buffer, atoms, format="cif")
    return buffer.getvalue().decode("utf-8")


def show_atoms_py3dmol(
    atoms: Atoms,
    *,
    width: int = 850,
    height: int = 520,
) -> None:
    display_atoms = atoms.repeat((2, 2, 1))

    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(atoms_to_cif_string(display_atoms), "cif")
    viewer.setStyle(
        {
            "sphere": {"scale": 0.32},
            "stick": {"radius": 0.14},
        }
    )
    viewer.addUnitCell()
    viewer.zoomTo()
    viewer.show()


structure_catalog = {
    "Clean eSEN-relaxed slab": str(clean_trajectory_path),
}

if not global_minima.empty:
    for row in global_minima.itertuples(index=False):
        label = (
            f"GLOBAL MIN | {row.adsorbate}* | {row.site_family} | "
            f"Eads = {row.eSEN_pred_ads_energy_eV:.4f} eV | {row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)

if not unique_site_minima.empty:
    for row in unique_site_minima.itertuples(index=False):
        label = (
            f"UNIQUE SITE | {row.adsorbate}* | {row.site_instance_key} | "
            f"Eads = {row.eSEN_pred_ads_energy_eV:.4f} eV | {row.config_id}"
        )
        structure_catalog[label] = str(row.trajectory)

selector = widgets.Dropdown(
    options=list(structure_catalog.keys()),
    value="Clean eSEN-relaxed slab",
    description="Structure:",
    layout=widgets.Layout(width="95%"),
)
viewer_output = widgets.Output()


def render_selected_structure(change=None) -> None:
    label = selector.value
    trajectory_path = Path(structure_catalog[label])

    with viewer_output:
        clear_output(wait=True)
        if not trajectory_path.exists():
            print("Structure file not found:", trajectory_path)
            return

        try:
            atoms = aseio.read(str(trajectory_path), index=-1)
            print(label)
            print("Visualization cell: 2 × 2 × 1")
            show_atoms_py3dmol(atoms)
        except Exception as exc:
            print(f"Could not display {label}")
            print(f"{type(exc).__name__}: {exc}")


selector.observe(render_selected_structure, names="value")
display(widgets.VBox([selector, viewer_output]))
render_selected_structure()


## 16. Repository-ready VASP single-point inputs

The final eSEN trajectory frame is exported for:

- the clean slab;
- the global O* minimum;
- the global OH* minimum; and
- the global OOH* minimum.

Each calculation directory contains:

- `POSCAR` with the final eSEN geometry;
- `INCAR` for a non-spin-polarized RPBE-D3 single-point calculation;
- `KPOINTS` generated from

$$
\left(\operatorname{round}\frac{40}{|\mathbf a|},
\operatorname{round}\frac{40}{|\mathbf b|},1\right);
$$

- `job.sh` as an editable Slurm template;
- `metadata.json` containing the model, source trajectory, adsorption energy,
  and DFT settings; and
- `POTCAR`, a simple non-runnable placeholder stating the required element
  order.

Replace the `POTCAR` placeholder privately with licensed PAW-PBE datasets before
submitting a calculation. Review the Slurm partition, module names, `NCORE`, and
`KPAR` for the target machine.


In [18]:
POTCAR_LABELS = {
    "H": "H",
    "O": "O",
    "Ti": "Ti",
    "Mn": "Mn",
    "Fe": "Fe",
    "Co": "Co",
    "Ni": "Ni",
    "Cu": "Cu",
    "Nb": "Nb",
    "Mo": "Mo",
    "Ru": "Ru",
    "Sn": "Sn",
    "Sb": "Sb",
    "Ta": "Ta",
    "Ir": "Ir",
    "Pt": "Pt",
}


def write_poscar_with_selective_dynamics(
    poscar_path: Path,
    atoms: Atoms,
    system_label: str,
) -> list[str]:
    """Write a deterministic VASP5 POSCAR and preserve FixAtoms flags."""
    symbols = atoms.get_chemical_symbols()

    element_order: list[str] = []
    for symbol in symbols:
        if symbol not in element_order:
            element_order.append(symbol)

    grouped_indices = [
        index
        for element in element_order
        for index, symbol in enumerate(symbols)
        if symbol == element
    ]
    counts = [
        sum(symbol == element for symbol in symbols)
        for element in element_order
    ]

    fixed_indices: set[int] = set()
    for constraint in getattr(atoms, "constraints", []):
        if isinstance(constraint, FixAtoms):
            fixed_indices.update(map(int, constraint.get_indices()))

    scaled_positions = atoms.get_scaled_positions(wrap=False)
    cell = np.asarray(atoms.cell.array, dtype=float)

    lines = [
        f"{system_label} | final eSEN/OC25 frame",
        "1.0",
    ]
    lines.extend(
        "  " + " ".join(f"{component:.12f}" for component in vector)
        for vector in cell
    )
    lines.append("  " + " ".join(element_order))
    lines.append("  " + " ".join(map(str, counts)))
    lines.append("Selective dynamics")
    lines.append("Direct")

    for index in grouped_indices:
        position = scaled_positions[index]
        flags = "F F F" if index in fixed_indices else "T T T"
        lines.append(
            "  "
            + " ".join(f"{component:.12f}" for component in position)
            + f"  {flags}"
        )

    poscar_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return element_order


def write_kpoints(directory: Path, atoms: Atoms) -> dict:
    """Write the reciprocal-length Monkhorst–Pack mesh."""
    cell = np.asarray(atoms.cell.array, dtype=float)
    a_length = float(np.linalg.norm(cell[0]))
    b_length = float(np.linalg.norm(cell[1]))

    if not np.isfinite(a_length) or not np.isfinite(b_length):
        raise ValueError("Non-finite in-plane lattice-vector length.")
    if a_length <= 0.0 or b_length <= 0.0:
        raise ValueError("Invalid in-plane lattice vectors for KPOINTS.")

    mesh = [
        max(1, int(round(KPOINT_DENSITY / a_length))),
        max(1, int(round(KPOINT_DENSITY / b_length))),
        1,
    ]

    path = directory / "KPOINTS"
    path.write_text(
        "Automatic mesh\n"
        "0\n"
        "Monkhorst-Pack\n"
        + " ".join(map(str, mesh))
        + "\n0 0 0\n",
        encoding="utf-8",
    )

    return {
        "path": str(path),
        "mesh": mesh,
        "a_length_A": a_length,
        "b_length_A": b_length,
        "density_parameter": KPOINT_DENSITY,
        "rule": "round(40/|a|), round(40/|b|), 1",
        "scheme": "Monkhorst-Pack",
    }


def write_incar(directory: Path, system_label: str) -> Path:
    """Write a non-spin-polarized RPBE-D3 single-point INCAR."""
    incar_text = f"""
SYSTEM = {system_label} | RPBE-D3 single point
PREC = {VASP_PREC}
ENCUT = {VASP_ENCUT_EV}
EDIFF = {VASP_EDIFF:.0E}

# Exchange-correlation functional and dispersion
GGA = RP
IVDW = {VASP_IVDW}
ISPIN = 1
NELM = {VASP_NELM}

# Electronic minimization
ALGO = All
ISMEAR = 0
SIGMA = 0.1
LREAL = Auto

# Single-point calculation: no ionic update
IBRION = -1
NSW = {VASP_NSW}
ISYM = 0

# Dipole correction along the surface normal
LDIPOL = .TRUE.
IDIPOL = 3
DIPOL = 0.5 0.5 0.5

# Output and numerical accuracy
LWAVE = .FALSE.
LCHARG = .TRUE.
LASPH = .TRUE.
AMIN = 0.01
ADDGRID = .TRUE.
LAECHG = .TRUE.

# Parallelization: review for the target machine
NCORE = {VASP_NCORE}
KPAR = {VASP_KPAR}
"""

    path = directory / "INCAR"
    path.write_text(textwrap.dedent(incar_text).strip() + "\n", encoding="utf-8")
    return path


def safe_job_name(label: str) -> str:
    """Convert a scientific label to a valid Slurm job name."""
    cleaned = re.sub(r"[^A-Za-z0-9_-]+", "-", label).strip("-")
    return cleaned[:64] or "vasp_sp"


def write_job_script(directory: Path, job_label: str) -> Path:
    """Write an editable Slurm template."""
    job_name = safe_job_name(job_label)

    job_text = f"""#!/bin/bash
#SBATCH -N 1
#SBATCH -n 64
#SBATCH --time=24:00:00
#SBATCH -p rome
#SBATCH -J {job_name}
#SBATCH --output=out.%j
#SBATCH --error=err.%j

set -euo pipefail

if grep -q "POTCAR NOT DISTRIBUTED" POTCAR; then
    echo "ERROR: Replace the POTCAR placeholder with a licensed POTCAR."
    exit 2
fi

module purge
module load 2024
module load VASP6/6.4.3-foss-2024a

srun vasp_std > vasp.out
"""

    path = directory / "job.sh"
    path.write_text(textwrap.dedent(job_text), encoding="utf-8")
    path.chmod(0o755)
    return path


def write_potcar_placeholder(
    directory: Path,
    element_order: list[str],
) -> tuple[Path, list[str]]:
    """Write one simple, non-runnable POTCAR placeholder."""
    potcar_labels = [POTCAR_LABELS[element] for element in element_order]
    path = directory / "POTCAR"
    path.write_text(
        "POTCAR NOT DISTRIBUTED\n\n"
        "Replace this placeholder with licensed PAW-PBE datasets in the "
        "following POSCAR order:\n"
        + " ".join(potcar_labels)
        + "\n",
        encoding="utf-8",
    )
    return path, potcar_labels


def export_vasp_calculation(
    *,
    atoms: Atoms,
    directory: Path,
    system_label: str,
    metadata: dict,
) -> dict:
    """Export one repository-safe VASP single-point calculation folder."""
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)

    poscar_path = directory / "POSCAR"
    element_order = write_poscar_with_selective_dynamics(
        poscar_path,
        atoms,
        system_label,
    )
    kpoint_metadata = write_kpoints(directory, atoms)
    incar_path = write_incar(directory, system_label)
    job_path = write_job_script(directory, f"{system_label}_RPBE_D3_SP")
    potcar_path, potcar_labels = write_potcar_placeholder(
        directory,
        element_order,
    )

    case_metadata = {
        **metadata,
        "system_label": system_label,
        "source_frame": -1,
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": element_order,
        "potcar_labels": potcar_labels,
        "kpoint_metadata": kpoint_metadata,
        "vasp_run_mode": VASP_RUN_MODE,
        "potcar_distributed": False,
    }

    metadata_path = directory / "metadata.json"
    metadata_path.write_text(
        json.dumps(case_metadata, indent=2, default=str) + "\n",
        encoding="utf-8",
    )

    return {
        "system_label": system_label,
        "directory": str(directory),
        "formula": atoms.get_chemical_formula(),
        "n_atoms": len(atoms),
        "element_order": " ".join(element_order),
        "potcar_labels": " ".join(potcar_labels),
        "kpoints": " ".join(map(str, kpoint_metadata["mesh"])),
        "a_length_A": kpoint_metadata["a_length_A"],
        "b_length_A": kpoint_metadata["b_length_A"],
        "kpoint_density_parameter": KPOINT_DENSITY,
        "POSCAR": str(poscar_path),
        "INCAR": str(incar_path),
        "KPOINTS": kpoint_metadata["path"],
        "job.sh": str(job_path),
        "POTCAR": str(potcar_path),
        "metadata.json": str(metadata_path),
    }


In [19]:
vasp_rows = []

dft_protocol = {
    "run_mode": VASP_RUN_MODE,
    "functional": "RPBE",
    "dispersion": "D3 zero damping",
    "IVDW": VASP_IVDW,
    "spin_polarized": False,
    "ISPIN": 1,
    "PREC": VASP_PREC,
    "ENCUT_eV": VASP_ENCUT_EV,
    "EDIFF_eV": VASP_EDIFF,
    "EDIFFG_eV_A": VASP_EDIFFG,
    "NELM": VASP_NELM,
    "IBRION": -1,
    "NSW": VASP_NSW,
    "dipole_correction": True,
    "IDIPOL": 3,
    "DIPOL": [0.5, 0.5, 0.5],
    "kpoint_scheme": "Monkhorst-Pack",
    "kpoint_density_parameter": KPOINT_DENSITY,
    "kpoint_rule": "round(40/|a|), round(40/|b|), 1",
}

vasp_rows.append(
    export_vasp_calculation(
        atoms=clean_final,
        directory=VASP_DIR / "bare",
        system_label=f"{selected_surface} bare",
        metadata={
            "surface": selected_surface,
            "bulk_id": BULK_ID,
            "miller_index": MILLER_INDEX,
            "surface_index": SURFACE_INDEX,
            "structure_type": "bare_slab",
            "model_name": MODEL_NAME,
            "source_trajectory": str(clean_trajectory_path),
            "ml_total_energy_eV": clean_energy_eV,
            "E_slab_eV": clean_energy_eV,
            "final_fmax_eV_A": clean_relaxation["final_fmax_eV_A"],
            "dft_protocol": dft_protocol,
        },
    )
)

available_adsorbates = set(
    global_minima.get("adsorbate", pd.Series(dtype=str)).tolist()
)
missing_adsorbates = set(ADSORBATE_NAMES) - available_adsorbates

if missing_adsorbates:
    print(
        "[WARN] No accepted global minimum was available for:",
        ", ".join(sorted(missing_adsorbates)),
    )

for adsorbate_name in ADSORBATE_NAMES:
    if global_minima.empty:
        continue

    selected = global_minima[
        global_minima["adsorbate"] == adsorbate_name
    ]
    if selected.empty:
        continue

    row = selected.iloc[0]
    final = aseio.read(str(row["trajectory"]), index=-1)

    vasp_rows.append(
        export_vasp_calculation(
            atoms=final,
            directory=VASP_DIR / adsorbate_name,
            system_label=f"{selected_surface} {adsorbate_name}",
            metadata={
                "surface": selected_surface,
                "bulk_id": BULK_ID,
                "miller_index": MILLER_INDEX,
                "surface_index": SURFACE_INDEX,
                "structure_type": "adsorbate_global_minimum",
                "adsorbate": adsorbate_name,
                "config_id": str(row["config_id"]),
                "placement_kind": str(row["placement_kind"]),
                "site_type": str(row["site_type"]),
                "site_family": str(row["site_family"]),
                "site_instance_key": str(row["site_instance_key"]),
                "neighbor_indices": str(row["neighbor_indices"]),
                "neighbor_composition": str(row["neighbor_composition"]),
                "model_name": MODEL_NAME,
                "source_trajectory": str(row["trajectory"]),
                "ml_total_energy_eV": float(row["ml_total_energy_eV"]),
                "E_adslab_eV": float(row["E_adslab_eV"]),
                "E_slab_eV": float(row["E_slab_eV"]),
                "adslab_minus_slab_eV": float(row["adslab_minus_slab_eV"]),
                "che_reference_energy_eV": float(
                    row["che_reference_energy_eV"]
                ),
                "eSEN_pred_ads_energy_eV": float(
                    row["eSEN_pred_ads_energy_eV"]
                ),
                "final_fmax_eV_A": float(row["final_fmax_eV_A"]),
                "energy_interpretation": (
                    "ml_total_energy_eV and E_adslab_eV are eSEN total "
                    "energies. eSEN_pred_ads_energy_eV equals E_adslab_eV "
                    "- E_slab_eV - che_reference_energy_eV."
                ),
                "dft_protocol": dft_protocol,
            },
        )
    )

vasp_manifest = pd.DataFrame(vasp_rows)
vasp_manifest.to_csv(VASP_DIR / "manifest.csv", index=False)
display(vasp_manifest)


,system_label,directory,formula,n_atoms,element_order,potcar_labels,kpoints,a_length_A,b_length_A,kpoint_density_parameter,POSCAR,INCAR,KPOINTS,job.sh,POTCAR,metadata.json
0,mp-10260_111 bare,/content/stageII_esen_oc25_results/mp-10260_11...,Ni36Sb12,48,Ni Sb,Ni Sb,5 5 1,8.466487,8.466487,40.0,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...
1,mp-10260_111 O,/content/stageII_esen_oc25_results/mp-10260_11...,Ni36OSb12,49,Ni Sb O,Ni Sb O,5 5 1,8.466487,8.466487,40.0,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...
2,mp-10260_111 OH,/content/stageII_esen_oc25_results/mp-10260_11...,HNi36OSb12,50,Ni Sb O H,Ni Sb O H,5 5 1,8.466487,8.466487,40.0,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...
3,mp-10260_111 OOH,/content/stageII_esen_oc25_results/mp-10260_11...,HNi36O2Sb12,51,Ni Sb O H,Ni Sb O H,5 5 1,8.466487,8.466487,40.0,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...,/content/stageII_esen_oc25_results/mp-10260_11...


## 17. Export results and create the reproducibility archive

The final cells write the screening, trajectory-validation, site-analysis,
selected-structure, and VASP manifests to CSV files. They also record the run
configuration in JSON, create a concise README, and package the complete result
directory as a ZIP archive for repository deposition or transfer to a DFT
computing environment.


In [20]:
def export_selected_poscars(
    table: pd.DataFrame,
    category: str,
) -> pd.DataFrame:
    exported = []

    if table.empty:
        return pd.DataFrame()

    for row in table.itertuples(index=False):
        final = aseio.read(
            row.trajectory,
            index=-1,
        )

        if category == "unique_site_minima":
            folder = (
                f"{row.config_id}__"
                f"{row.site_instance_key}"
            )
        else:
            folder = (
                f"{row.adsorbate}_global_min__"
                f"{row.config_id}"
            )

        folder = folder.replace("/", "_")

        path = write_poscar(
            final,
            POSCAR_DIR
            / category
            / row.adsorbate
            / folder,
        )

        exported.append(
            {
                "category": category,
                "adsorbate": row.adsorbate,
                "config_id": row.config_id,
                "site_type": row.site_type,
                "site_family": row.site_family,
                "site_instance_key": (
                    row.site_instance_key
                ),
                "ml_total_energy_eV": (
                    row.ml_total_energy_eV
                ),
                "E_adslab_eV": row.E_adslab_eV,
                "E_slab_eV": row.E_slab_eV,
                "che_reference_energy_eV": (
                    row.che_reference_energy_eV
                ),
                "eSEN_pred_ads_energy_eV": (
                    row.eSEN_pred_ads_energy_eV
                ),
                "trajectory": row.trajectory,
                "poscar": str(path),
            }
        )

    return pd.DataFrame(exported)


selected_poscars = pd.concat(
    [
        export_selected_poscars(
            unique_site_minima,
            "unique_site_minima",
        ),
        export_selected_poscars(
            global_minima,
            "global_minima",
        ),
    ],
    ignore_index=True,
)

tables = {
    "adsorbate_summary.csv": adsorbate_summary,
    "placement_summary.csv": placement_summary,
    "initial_configuration_metadata.csv": (
        configuration_table
    ),
    "clean_relaxation.csv": (
        clean_relaxation_table
    ),
    "relaxation_status.csv": (
        relaxation_status
    ),
    "trajectory_validation.csv": (
        trajectory_validation
    ),
    "all_adsorption_sites.csv": (
        all_adsorption_sites
    ),
    "unique_site_minima.csv": (
        unique_site_minima
    ),
    "site_family_minima.csv": (
        site_family_minima
    ),
    "global_minima.csv": global_minima,
    "selected_poscars.csv": selected_poscars,
    "vasp_manifest.csv": vasp_manifest,
    "molecular_reference_energies.csv": molecular_reference_energies,
    "che_reference_energies.csv": che_reference_table,
    "esen_global_adsorption_energies.csv": global_adsorption_energies,
}

for filename, table in tables.items():
    table.to_csv(
        TABLE_DIR / filename,
        index=False,
    )

run_metadata = {
    "workflow_stage": (
        "Stage II eSEN/OC25 total-energy relaxation, electronic CHE "
        "adsorption-energy screening, and DFT input generation"
    ),
    "surface": selected_surface,
    "bulk_id": BULK_ID,
    "miller_index": MILLER_INDEX,
    "surface_index": SURFACE_INDEX,
    "model_name": MODEL_NAME,
    "random_sites_per_adsorbate": (
        RANDOM_SITES_PER_ADSORBATE
    ),
    "fmax_threshold_eV_A": (
        FMAX_THRESHOLD
    ),
    "max_relax_steps": MAX_RELAX_STEPS,
    "adsorbates": list(ADSORBATE_NAMES),
    "energy_interpretation": (
        "eSEN predicts total energies for the independently relaxed bare "
        "slab, adsorbate-slab structures, H2, and H2O. Electronic CHE "
        "adsorption energies are E_ads+slab - E_slab - E_reference. "
        "No generic ZPE, entropy, solvation, limiting-potential, or "
        "overpotential correction is applied in Stage II."
    ),
    "adsorption_energy_formula": (
        "eSEN_pred_ads_energy_eV = E_adslab_eV - E_slab_eV "
        "- che_reference_energy_eV"
    ),
    "bare_slab_total_energy_eV": clean_energy_eV,
    "molecular_reference_energies_eV": {
        "H2": E_H2_EV,
        "H2O": E_H2O_EV,
    },
    "che_reference_energies_eV": CHE_REFERENCE_ENERGIES_EV,
    "vasp_protocol": {
        "functional": "RPBE",
        "dispersion": "D3 zero damping",
        "IVDW": VASP_IVDW,
        "ISPIN": 1,
        "spin_polarized": False,
        "PREC": VASP_PREC,
        "ENCUT_eV": VASP_ENCUT_EV,
        "EDIFF_eV": VASP_EDIFF,
        "EDIFFG_eV_A": VASP_EDIFFG,
        "IBRION": -1,
        "NSW": VASP_NSW,
        "LDIPOL": True,
        "IDIPOL": 3,
        "DIPOL": [0.5, 0.5, 0.5],
        "kpoint_scheme": "Monkhorst-Pack",
        "kpoint_density_parameter": (
            KPOINT_DENSITY
        ),
        "kpoint_rule": (
            "round(40/|a|), "
            "round(40/|b|), 1"
        ),
    },
    "potcar_distributed": False,
}

(
    SYSTEM_DIR / "run_metadata.json"
).write_text(
    json.dumps(
        run_metadata,
        indent=2,
        default=str,
    )
    + "\n",
    encoding="utf-8",
)

readme_text = f"""# Stage II eSEN/OC25 reproducibility results

Surface: `{selected_surface}`
Model: `{MODEL_NAME}`
Force threshold: `{FMAX_THRESHOLD} eV/Å`

## Scientific scope

This directory contains the Stage II eSEN/OC25 total-energy relaxation,
trajectory validation, adsorption-site classification, unique-site screening,
electronic CHE adsorption energies, and VASP input generation for one Stage I
candidate.

The bare slab is relaxed first. For every adsorbate configuration, eSEN predicts
the relaxed adsorbate+slab total energy and the adsorption energy is calculated
as

`eSEN_pred_ads_energy_eV = E_adslab_eV - E_slab_eV - che_reference_energy_eV`

Stage II does not apply generic ZPE, entropy,
solvation, limiting-potential, or overpotential corrections.

## Main tables

- `tables/clean_relaxation.csv`
- `tables/molecular_reference_energies.csv`
- `tables/che_reference_energies.csv`
- `tables/relaxation_status.csv`
- `tables/trajectory_validation.csv`
- `tables/all_adsorption_sites.csv`
- `tables/unique_site_minima.csv`
- `tables/site_family_minima.csv`
- `tables/global_minima.csv`
- `tables/esen_global_adsorption_energies.csv`
- `vasp/manifest.csv`

## VASP directories

- `vasp/bare`
- `vasp/O`
- `vasp/OH`
- `vasp/OOH`

The VASP inputs use non-spin-polarized RPBE-D3 single-point calculations
with a z-directed dipole correction. The Monkhorst–Pack mesh is generated as

`round(40/|a|) round(40/|b|) 1`

using the in-plane lattice-vector lengths in ångström.

Every `POTCAR` is an intentional non-runnable placeholder. Replace it
privately with licensed PAW-PBE datasets in the stated element order.
"""

(
    SYSTEM_DIR / "README.md"
).write_text(
    textwrap.dedent(readme_text),
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    base_name=str(
        ROOT_DIR / f"{SYSTEM_ID}_stageII"
    ),
    format="zip",
    root_dir=str(ROOT_DIR),
    base_dir=SYSTEM_ID,
)

print("Result directory:", SYSTEM_DIR)
print("ZIP archive:", archive_path)
print("\nGenerated VASP meshes:")
if not vasp_manifest.empty:
    display(
        vasp_manifest[
            [
                "system_label",
                "a_length_A",
                "b_length_A",
                "kpoints",
            ]
        ]
    )


Result directory: /content/stageII_esen_oc25_results/mp-10260_111_term0_esen_oc25
ZIP archive: /content/stageII_esen_oc25_results/mp-10260_111_term0_esen_oc25_stageII.zip

Generated VASP meshes:


,system_label,a_length_A,b_length_A,kpoints
0,mp-10260_111 bare,8.466487,8.466487,5 5 1
1,mp-10260_111 O,8.466487,8.466487,5 5 1
2,mp-10260_111 OH,8.466487,8.466487,5 5 1
3,mp-10260_111 OOH,8.466487,8.466487,5 5 1


In [21]:
import sys

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive_path)
else:
    print("Archive available at:", archive_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reproducibility notes and next workflow stage

The principal output column is `eSEN_pred_ads_energy_eV`. For every accepted
configuration,

$$
E_{\mathrm{ads}}^{\mathrm{eSEN/CHE}}
=E_{\mathrm{ads+slab}}-E_{\mathrm{slab}}-E_{\mathrm{reference}}.
$$

The accompanying total-energy columns are retained only to make this
subtraction transparent and reproducible.

This notebook is the public demonstration of the **Stage II eSEN/OC25 branch**.
The full study used the same workflow for 24 top candidates identified in
Stage I, while one slab is selected here as a complete executable example.

The exported bare, O*, OH*, and OOH* folders are prepared for RPBE-D3
single-point calculations. Those DFT energies are used to validate the eSEN
screening predictions before system-specific thermodynamic and solvation
corrections are applied.

The next notebook performs the corresponding Stage II screening with the
**AQCat model**, enabling a consistent comparison of the two machine-learning
branches before the final DFT analysis.


## References

1. Nørskov, J. K.; Rossmeisl, J.; Logadottir, A.; Lindqvist, L.; Kitchin,
   J. R.; Bligaard, T.; Jónsson, H.  
   **Origin of the Overpotential for Oxygen Reduction at a Fuel-Cell
   Cathode.** *J. Phys. Chem. B* **2004**, *108*, 17886–17892.
   DOI: `10.1021/jp047349j`.

2. Sahoo, S. J.; Maraschin, M.; Levine, D. S.; *et al.*  
   **The Open Catalyst 2025 (OC25) Dataset and Models for Solid–Liquid
   Interfaces.** *arXiv* **2025**, 2509.17862.

3. Fu, X.; Wood, B. M.; Barroso-Luque, L.; Levine, D. S.; Gao, M.;
   Dzamba, M.; Zitnick, C. L.  
   **Learning Smooth and Expressive Interatomic Potentials for Physical
   Property Prediction.** *Proceedings of Machine Learning Research*
   **2025**, *267*, 17875–17893.

4. Lan, J.; Palizhati, A.; Shuaibi, M.; *et al.*  
   **AdsorbML: A Leap in Efficiency for Adsorption Energy Calculations Using
   Generalizable Machine Learning Potentials.**
   *npj Computational Materials* **2023**, *9*, 172.

5. FAIR Chemistry OC25 model card and checkpoint documentation:
   `https://huggingface.co/facebook/OC25`.
